# FII / DII Trendlyne Investment Pattern Analysis

### What this notebook does
1. Scrapes all investor links from Trendlyne (Indian + US)
2. Fetches quarterly net-worth history per investor
3. Computes full professional metrics vs benchmark (Nifty 50 for India, S&P 500 for US)
4. Adds `good_*` YES/NO flags for every paired metric — easy Google Sheet filtering
5. Generates `signal`, `interpretation`, `reason`, `confidence_score`
6. Writes comprehensive glossary to `fii_dii_metric_docs`

### Output tabs
| Tab | Content |
|---|---|
| `indian_fii_dii_trend_lyne_links` | type \| name \| links (Indian investors) |
| `fii_dii_indian_investment_summary` | Full metrics vs Nifty 50 |
| `us_fii_dii_trend_lyne_links` | type \| name \| links (US investors) |
| `fii_dii_us_investment_summary` | Full metrics vs S&P 500 |
| `fii_dii_metric_docs` | Glossary — every metric explained |

### Column format
`type | name | links | signal | good_* flags | nifty/sp500_metric | investor_metric | ...`

### Run order
**India:** Cell 1→5 (setup), Cell 6 (scrape), Cell 7 (Nifty data), Cell 8 (metrics)
**US:**    Cell 9 (scrape), Cell 10 (S&P 500 data), Cell 11 (metrics)
**Docs:**  Cell 12 (glossary)


In [1]:
# ── CELL 1: IMPORTS ──────────────────────────────────────────────────────
import json, os, re, time, warnings
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
from datetime import datetime
from scipy.optimize import brentq
from scipy import stats as sp_stats
import gspread
from google.oauth2.service_account import Credentials
from gspread_dataframe import get_as_dataframe, set_with_dataframe
warnings.filterwarnings('ignore')
print('All imports OK.')

All imports OK.


In [2]:
# ── CELL 2: CONFIGURATION ────────────────────────────────────────────────
SHEET_KEY = '1rIFmhm37XEJsfXV2Nn1QPfakLG9xvMmEsjJ7YYule8g'

# Indian settings
INDIA_BENCHMARK = '^NSEI'
INDIA_RFR       = 0.065    # India 10-yr G-Sec yield

# US settings
US_BENCHMARK    = '^GSPC'
US_RFR          = 0.045    # US 10-yr Treasury yield

REQUEST_DELAY   = 1.5      # seconds between Trendlyne fetches
MIN_QUARTERS    = 5        # skip investors with fewer data points

REQUEST_HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'}
MONTH_MAP = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
LAST_DAY  = {1:31,2:28,3:31,4:30,5:31,6:30,7:31,8:31,9:30,10:31,11:30,12:31}

import json as _j, os as _o
_raw = _o.environ.get('GCP_SERVICE_ACCOUNT')
if _raw:
    SERVICE_ACCOUNT_INFO = _j.loads(_raw)
    print('Credentials loaded from env GCP_SERVICE_ACCOUNT.')
else:
    # Local / interactive run: fall back to the gitignored service_account.json in the repo folder.
    # CI sets the env var; this just lets the notebook run on a laptop with no extra setup.
    if not _o.path.exists('service_account.json'):
        raise RuntimeError('No credentials found: set the GCP_SERVICE_ACCOUNT env var, OR run this '
                           'notebook from the repo folder that contains service_account.json '
                           '(it is gitignored, so it never leaves your machine).')
    with open('service_account.json') as _f:
        SERVICE_ACCOUNT_INFO = _j.load(_f)
    print('Credentials loaded from local service_account.json.')
print('Config ready.')

Credentials loaded from local service_account.json.
Config ready.


In [3]:
# ── CELL 3: GOOGLE SHEETS UTILITIES ──────────────────────────────────────

def _gsheet_client():
    creds = Credentials.from_service_account_info(
        SERVICE_ACCOUNT_INFO,
        scopes=['https://www.googleapis.com/auth/spreadsheets',
                'https://www.googleapis.com/auth/drive']
    )
    return gspread.authorize(creds)

def create_tab(sheet_key, tab_name, rows=2000, cols=60):
    book = _gsheet_client().open_by_key(sheet_key)
    try:
        book.worksheet(tab_name); print(f'  Tab "{tab_name}" exists.')
    except gspread.exceptions.WorksheetNotFound:
        book.add_worksheet(title=tab_name, rows=rows, cols=cols)
        print(f'  Created "{tab_name}".')

def read_sheet(sheet_key, tab_name):
    try:
        ws = _gsheet_client().open_by_key(sheet_key).worksheet(tab_name)
        return get_as_dataframe(ws, evaluate_formulas=True).dropna(how='all')
    except gspread.exceptions.WorksheetNotFound:
        print(f'  Tab "{tab_name}" not found.'); return pd.DataFrame()
    except Exception as e:
        print(f'  read_sheet error: {e}'); return pd.DataFrame()

def read_skip_lines(sheet_key, tab_name, n):
    df = read_sheet(sheet_key, tab_name)
    return df.iloc[n:].reset_index(drop=True) if not df.empty else df

def clear_sheet(sheet_key, tab_name):
    try:
        _gsheet_client().open_by_key(sheet_key).worksheet(tab_name).clear()
        print(f'  Cleared "{tab_name}".')
    except gspread.exceptions.WorksheetNotFound:
        print(f'  Tab "{tab_name}" not found.')

def write_sheet(sheet_key, tab_name, df):
    """Skip write entirely if df is empty — preserves existing data on scrape failure."""
    if df is None or df.empty:
        print(f'  SKIPPED "{tab_name}" — empty df (existing data preserved).'); return
    try:
        book = _gsheet_client().open_by_key(sheet_key)
        try: ws = book.worksheet(tab_name)
        except gspread.exceptions.WorksheetNotFound:
            ws = book.add_worksheet(title=tab_name, rows=df.shape[0]+20, cols=df.shape[1]+5)
        ws.clear()
        set_with_dataframe(ws, df.fillna(''))
        print(f'  Wrote {len(df)} rows to "{tab_name}".')
    except Exception as e:
        print(f'  write_sheet error "{tab_name}": {e}')

print('GSheet utilities ready.')

GSheet utilities ready.


In [4]:
# ── CELL 4: TRENDLYNE SCRAPING FUNCTIONS ─────────────────────────────────

def _valid_link(href, text, path_fragment):
    if not href or path_fragment not in href: return False
    if any(k in href for k in ['/index/', '/#', '/visitor/']): return False
    if not text or len(text.strip()) < 3: return False
    blocked = {'superstars','more','login / sign up','superstar portfolios',
               'us superstars','individual investors','institutional investors','fiis'}
    return text.strip().lower() not in blocked

_BROWSER_HEADERS = {
    'User-Agent':      ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                        '(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'),
    'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection':      'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest':  'document',
    'Sec-Fetch-Mode':  'navigate',
    'Sec-Fetch-Site':  'none',
    'Sec-Fetch-User':  '?1',
}

def _make_session(base_url):
    """Warm-up session: visit homepage to acquire cookies before scraping sub-pages."""
    s = requests.Session()
    s.headers.update(_BROWSER_HEADERS)
    try:
        time.sleep(REQUEST_DELAY)
        s.get(base_url, timeout=20)
    except Exception:
        pass  # best-effort cookie warm-up
    return s

def scrape_investor_links(url, investor_type, base_url, path_fragment, session=None):
    """
    Generic scraper for both Indian and US Trendlyne pages.
    base_url      : 'https://trendlyne.com' or 'https://us.trendlyne.com'
    path_fragment : '/portfolio/superstar-shareholders/' or '/us/portfolio/superstar-shareholders/'
    session       : optional requests.Session (required for US pages that block plain requests)
    Returns DataFrame[type, name, links]. Returns empty df on failure.
    """
    print(f'  Scraping {investor_type} ...')
    try:
        time.sleep(REQUEST_DELAY)
        if session:
            resp = session.get(url, timeout=20)
        else:
            resp = requests.get(url, headers=REQUEST_HEADERS, timeout=20)
        resp.raise_for_status()
    except Exception as e:
        print(f'  ERROR: {e}'); return pd.DataFrame()
    soup = BeautifulSoup(resp.text, 'html.parser')
    rows = []
    for a in soup.find_all('a'):
        href = a.get('href',''); text = a.get_text(strip=True)
        if not _valid_link(href, text, path_fragment): continue
        if href.startswith('/'): href = base_url + href
        rows.append({'type': investor_type, 'name': text.strip().lower(), 'links': href})
    df = pd.DataFrame(rows).drop_duplicates(subset=['links']).reset_index(drop=True)
    print(f'  Found {len(df)} investors.')
    return df

# Convenience wrappers
def scrape_india(url, investor_type):
    return scrape_investor_links(url, investor_type,
                                  'https://trendlyne.com',
                                  '/portfolio/superstar-shareholders/')

def scrape_us(url, investor_type, session=None):
    return scrape_investor_links(url, investor_type,
                                  'https://us.trendlyne.com',
                                  '/us/portfolio/superstar-shareholders/',
                                  session=session)

print('Scraping functions ready.')

Scraping functions ready.


In [5]:
# ── CELL 5: PROFESSIONAL METRICS ENGINE ──────────────────────────────────

# ── Low-level helpers ─────────────────────────────────────────────────────

def _parse_quarter_label(label):
    label = label.strip()
    m = re.match(r'([A-Za-z]+)\s*(\d{4})', label)
    if not m: return None
    month = MONTH_MAP.get(m.group(1)[:3].title())
    if not month: return None
    year = int(m.group(2)); day = LAST_DAY.get(month, 30)
    if month == 2 and (year % 400 == 0 or (year % 100 != 0 and year % 4 == 0)): day = 29
    return datetime(year, month, day)

def fetch_portfolio_history(url, session=None, retries=4):
    """Fetch quarterly net-worth history from Trendlyne. Returns (df, status):
      'ok'    - chart node found and parsed to >=1 quarter,
      'empty' - page rendered (200) but the investor genuinely has NO net-worth chart / all-null data,
      'fail'  - non-200 / exception on EVERY retry (i.e. throttled or blocked).
    Only a real block ('fail') is retried with back-off; a 200 page with no chart is a legitimate
    'empty' and returns immediately, so a site-wide block can't turn every chart-less investor into
    4x retries (which would blow the CI budget). Pass a warmed `session` to cut CI/datacenter blocks."""
    last = None
    for attempt in range(retries):
        try:
            time.sleep(REQUEST_DELAY + np.random.uniform(0, 1.5))
            resp = session.get(url, timeout=20) if session is not None \
                else requests.get(url, headers=REQUEST_HEADERS, timeout=20)
            if resp.status_code != 200:                          # throttled/blocked -> back off + retry
                last = f'HTTP {resp.status_code}'
                if attempt < retries - 1: time.sleep(6 * (attempt + 1))
                continue
            node = BeautifulSoup(resp.text, 'html.parser').select_one('#highchart-superstar-shareholder-net-worth')
            if node is None:
                return pd.DataFrame(), 'empty'                   # page rendered, no chart = chart-less investor
            raw = node.get('data-jsondata', '')
            if not raw:
                return pd.DataFrame(), 'empty'                   # node present but no data
            data = json.loads(raw.replace('&quot;', '"'))
            rows = []
            for item in data:
                if not isinstance(item, (list, tuple)) or len(item) < 2 or item[1] is None: continue
                date = _parse_quarter_label(item[0])
                if date: rows.append({'period_date': date, 'net_worth_cr': float(item[1])})
            if not rows:
                return pd.DataFrame(), 'empty'                   # all-null / unparseable = no usable history
            return pd.DataFrame(rows).sort_values('period_date').reset_index(drop=True), 'ok'
        except Exception as e:
            last = e
            if attempt < retries - 1: time.sleep(6 * (attempt + 1))
    print(f'    fetch failed after {retries} tries ({url[-45:]}): {last}')
    return pd.DataFrame(), 'fail'


def _bm_on_or_before(series, date):
    valid = series[series.index <= pd.Timestamp(date)]
    return float(valid.iloc[-1]) if not valid.empty else np.nan

def _nan(v):
    try: return v != v
    except: return True

def _max_drawdown(values):
    arr = np.array([v for v in values if not _nan(v)], dtype=float)
    if len(arr) < 2: return np.nan
    peak = np.maximum.accumulate(arr)
    return float(np.min((arr - peak) / peak * 100))

def _peak_to_current(values):
    arr = np.array([v for v in values if not _nan(v)], dtype=float)
    if len(arr) < 1: return np.nan
    return float((arr[-1] - arr.max()) / arr.max() * 100)

def _max_consec_losses(rets):
    streak = best = 0
    for r in rets:
        streak = streak + 1 if r < 0 else 0
        best   = max(best, streak)
    return int(best)

def _xirr(start_date, end_date, sv, ev):
    try:
        years = (end_date - start_date).days / 365.25
        if years <= 0 or sv <= 0: return np.nan
        def npv(r): return -sv + ev / (1 + r) ** years
        return round(brentq(npv, -0.9999, 50.0) * 100, 2)
    except: return np.nan

def _r(v, d=2):
    try:
        f = float(v); return round(f, d) if not _nan(f) else ''
    except: return ''


# ── Core metrics computation ──────────────────────────────────────────────

def compute_metrics(portfolio_df, benchmark_series, risk_free_rate, bm='benchmark'):
    full = portfolio_df.sort_values('period_date').reset_index(drop=True)
    if len(full) < MIN_QUARTERS: return {}

    full['bm_px'] = full['period_date'].apply(lambda d: _bm_on_or_before(benchmark_series, d))
    df = full.dropna(subset=['bm_px']).copy()
    df['port_ret'] = df['net_worth_cr'].pct_change()
    df['bm_ret']   = df['bm_px'].pct_change()
    df = df.dropna(subset=['port_ret', 'bm_ret'])
    if len(df) < 4: return {}

    pr = df['port_ret'].values; br = df['bm_ret'].values; n = len(pr)
    bm_px_arr = np.array([v for v in full['bm_px'].values if not _nan(v)], dtype=float)

    ann_p  = (np.prod(1+pr)**(4/n)-1)*100;  ann_b  = (np.prod(1+br)**(4/n)-1)*100
    tot_p  = (full.iloc[-1]['net_worth_cr']/full.iloc[0]['net_worth_cr']-1)*100
    b0 = _bm_on_or_before(benchmark_series, full.iloc[0]['period_date'])
    b1 = _bm_on_or_before(benchmark_series, full.iloc[-1]['period_date'])
    tot_b  = (b1/b0-1)*100 if b0 > 0 and not _nan(b0) else np.nan
    xirr_p = _xirr(full.iloc[0]['period_date'], full.iloc[-1]['period_date'], full.iloc[0]['net_worth_cr'], full.iloc[-1]['net_worth_cr'])
    xirr_b = _xirr(full.iloc[0]['period_date'], full.iloc[-1]['period_date'], b0, b1) if not _nan(b0) else np.nan
    r1y_p  = (np.prod(1+pr[-4:]) -1)*100 if n >= 4  else np.nan
    r1y_b  = (np.prod(1+br[-4:]) -1)*100 if n >= 4  else np.nan
    r3y_p  = (np.prod(1+pr[-12:])-1)*100 if n >= 12 else np.nan
    r3y_b  = (np.prod(1+br[-12:])-1)*100 if n >= 12 else np.nan

    wr_p=float(np.mean(pr>0))*100; wr_b=float(np.mean(br>0))*100
    avg_p=float(np.mean(pr))*100;  avg_b=float(np.mean(br))*100
    med_p=float(np.median(pr))*100;med_b=float(np.median(br))*100

    vol_p=np.std(pr,ddof=1)*np.sqrt(4); vol_b=np.std(br,ddof=1)*np.sqrt(4)
    pr_neg=pr[pr<0]; br_neg=br[br<0]
    dvol_p=np.std(pr_neg,ddof=1)*np.sqrt(4) if len(pr_neg)>1 else np.nan
    dvol_b=np.std(br_neg,ddof=1)*np.sqrt(4) if len(br_neg)>1 else np.nan

    sh_p =(ann_p/100-risk_free_rate)/vol_p  if vol_p >0 else np.nan
    sh_b =(ann_b/100-risk_free_rate)/vol_b  if vol_b >0 else np.nan
    so_p =(ann_p/100-risk_free_rate)/dvol_p if not _nan(dvol_p) and dvol_p>0 else np.nan
    so_b =(ann_b/100-risk_free_rate)/dvol_b if not _nan(dvol_b) and dvol_b>0 else np.nan
    mdd_p=_max_drawdown(full['net_worth_cr'].values); mdd_b=_max_drawdown(bm_px_arr)
    cal_p=(ann_p/100)/abs(mdd_p/100) if mdd_p!=0 and not _nan(mdd_p) else np.nan
    cal_b=(ann_b/100)/abs(mdd_b/100) if mdd_b!=0 and not _nan(mdd_b) else np.nan
    p2c_p=_peak_to_current(full['net_worth_cr'].values); p2c_b=_peak_to_current(bm_px_arr)
    mcl_p=_max_consec_losses(pr); mcl_b=_max_consec_losses(br)

    skew_p=float(sp_stats.skew(pr));  skew_b=float(sp_stats.skew(br))
    kurt_p=float(sp_stats.kurtosis(pr)); kurt_b=float(sp_stats.kurtosis(br))

    beta, alpha_q = np.polyfit(br, pr, 1); alpha_ann = alpha_q*4*100
    corr = float(np.corrcoef(pr,br)[0,1])
    active=pr-br; te=float(np.std(active,ddof=1)*np.sqrt(4)*100)
    ir=float(np.mean(active)/np.std(active,ddof=1)*np.sqrt(4)) if np.std(active,ddof=1)>0 else np.nan
    outperf='YES' if alpha_ann>0 and not _nan(sh_p) and not _nan(sh_b) and sh_p>sh_b else 'NO'

    p = bm
    return {
        'data_from':             full.iloc[0]['period_date'].strftime('%Y-%m-%d'),
        'data_to':               full.iloc[-1]['period_date'].strftime('%Y-%m-%d'),
        'quarters_tracked':      len(full),
        'current_net_worth_cr':  _r(full.iloc[-1]['net_worth_cr']),
        f'{p}_ann_return_pct':          _r(ann_b),  'ann_return_pct':          _r(ann_p),
        f'{p}_total_return_pct':        _r(tot_b),  'total_return_pct':        _r(tot_p),
        f'{p}_xirr_approx_pct':         _r(xirr_b), 'xirr_approx_pct':         _r(xirr_p),
        f'{p}_rolling_1y_pct':          _r(r1y_b),  'rolling_1y_pct':          _r(r1y_p),
        f'{p}_rolling_3y_pct':          _r(r3y_b),  'rolling_3y_pct':          _r(r3y_p),
        f'{p}_win_rate_pct':            _r(wr_b),   'win_rate_pct':            _r(wr_p),
        f'{p}_avg_qtr_return_pct':      _r(avg_b),  'avg_qtr_return_pct':      _r(avg_p),
        f'{p}_median_qtr_return_pct':   _r(med_b),  'median_qtr_return_pct':   _r(med_p),
        f'{p}_volatility_ann_pct':      _r(vol_b*100),     'volatility_ann_pct':      _r(vol_p*100),
        f'{p}_downside_vol_ann_pct':    _r(dvol_b*100 if not _nan(dvol_b) else np.nan),
                                                            'downside_vol_ann_pct':    _r(dvol_p*100 if not _nan(dvol_p) else np.nan),
        f'{p}_sharpe_ratio':            _r(sh_b,3), 'sharpe_ratio':            _r(sh_p,3),
        f'{p}_sortino_ratio':           _r(so_b,3), 'sortino_ratio':           _r(so_p,3),
        f'{p}_calmar_ratio':            _r(cal_b,3),'calmar_ratio':            _r(cal_p,3),
        f'{p}_max_drawdown_pct':        _r(mdd_b),  'max_drawdown_pct':        _r(mdd_p),
        f'{p}_peak_to_current_pct':     _r(p2c_b),  'peak_to_current_pct':     _r(p2c_p),
        f'{p}_max_consec_losses':       int(mcl_b), 'max_consec_losses':       int(mcl_p),
        f'{p}_skew':                    _r(skew_b,3),'skew':                   _r(skew_p,3),
        f'{p}_kurtosis':                _r(kurt_b,3),'kurtosis':               _r(kurt_p,3),
        'alpha_ann_pct':         _r(alpha_ann),
        'beta':                  _r(beta,3),
        'correlation_to_bm':     _r(corr,3),
        'tracking_error_pct':    _r(te),
        'information_ratio':     _r(ir,3),
        f'is_outperforming_{p}': outperf,
    }


# ── good_* comparison flags ───────────────────────────────────────────────

def compute_comparison_flags(metrics, bm='benchmark'):
    def _cmp(pk, nk, hib=True):
        try:
            p = float(metrics.get(pk, '')); n = float(metrics.get(nk, ''))
            if _nan(p) or _nan(n): return ''
            return 'YES' if (p > n if hib else p < n) else 'NO'
        except: return ''
    flags = {
        'good_return'        : _cmp('ann_return_pct',        f'{bm}_ann_return_pct'),
        'good_total_return'  : _cmp('total_return_pct',      f'{bm}_total_return_pct'),
        'good_xirr'          : _cmp('xirr_approx_pct',       f'{bm}_xirr_approx_pct'),
        'good_rolling_1y'    : _cmp('rolling_1y_pct',        f'{bm}_rolling_1y_pct'),
        'good_rolling_3y'    : _cmp('rolling_3y_pct',        f'{bm}_rolling_3y_pct'),
        'good_win_rate'      : _cmp('win_rate_pct',          f'{bm}_win_rate_pct'),
        'good_avg_return'    : _cmp('avg_qtr_return_pct',    f'{bm}_avg_qtr_return_pct'),
        'good_sharpe'        : _cmp('sharpe_ratio',          f'{bm}_sharpe_ratio'),
        'good_sortino'       : _cmp('sortino_ratio',         f'{bm}_sortino_ratio'),
        'good_calmar'        : _cmp('calmar_ratio',          f'{bm}_calmar_ratio'),
        'good_volatility'    : _cmp('volatility_ann_pct',    f'{bm}_volatility_ann_pct',   hib=False),
        'good_downside_vol'  : _cmp('downside_vol_ann_pct',  f'{bm}_downside_vol_ann_pct', hib=False),
        'good_drawdown'      : _cmp('max_drawdown_pct',      f'{bm}_max_drawdown_pct'),
        'good_peak_to_curr'  : _cmp('peak_to_current_pct',   f'{bm}_peak_to_current_pct'),
        'good_consec_losses' : _cmp('max_consec_losses',     f'{bm}_max_consec_losses',    hib=False),
    }
    yes = sum(1 for v in flags.values() if v == 'YES')
    tot = sum(1 for v in flags.values() if v in ('YES','NO'))
    flags['score_vs_benchmark'] = f'{yes}/{tot}' if tot > 0 else ''
    flags['good_overall']       = 'YES' if tot > 0 and yes/tot >= 0.6 else ('NO' if tot > 0 else '')
    return flags


# ── Signal + Interpretation ───────────────────────────────────────────────

def make_interpretation_row(metrics, bm='benchmark'):
    def _f(k):
        v = metrics.get(k,'')
        try: f=float(v); return 0.0 if _nan(f) else f
        except: return 0.0
    alpha=_f('alpha_ann_pct'); sh_p=_f('sharpe_ratio'); sh_b=_f(f'{bm}_sharpe_ratio')
    mdd=_f('max_drawdown_pct'); cal=_f('calmar_ratio'); qtrs=int(metrics.get('quarters_tracked') or 0)
    if   alpha > 5  and sh_p > 0.8 and mdd > -40:          signal = 'STRONG BUY'
    elif alpha > 0  and sh_p > sh_b and cal  > 0.3:         signal = 'BUY'
    elif alpha > 0  and sh_p > sh_b:                        signal = 'WATCH'
    elif alpha < 0  or sh_p < 0    or mdd < -75:            signal = 'AVOID'
    else:                                                    signal = 'HOLD'
    if mdd < -70 and signal == 'STRONG BUY': signal = 'BUY'
    if mdd < -70 and signal == 'BUY':        signal = 'WATCH'
    r1 = f'alpha {alpha:+.1f}%'
    r2 = f'Sharpe {sh_p:.2f} > BM {sh_b:.2f}' if sh_p > sh_b else f'Sharpe {sh_p:.2f} < BM {sh_b:.2f}'
    r3 = f'MDD {mdd:.1f}%' if mdd < -60 else ''
    reason = ' | '.join(x for x in [r1,r2,r3] if x)[:100]
    perf = (f'Strong outperformer (alpha {alpha:+.1f}%)' if alpha > 10 else
            f'Moderate outperformer (alpha {alpha:+.1f}%)' if alpha > 0 else
            f'Underperforms benchmark (alpha {alpha:+.1f}%)')
    risk = ('excellent risk efficiency' if sh_p > 0.8 else 'decent risk/reward' if sh_p > 0.4 else 'poor risk/reward')
    dd   = ('low drawdown' if mdd > -40 else 'moderate drawdown' if mdd > -60 else 'high drawdown risk')
    cav  = f'; small sample ({qtrs} qtrs)' if qtrs < 8 else ''
    interpretation = f'{perf}, {risk}, {dd}{cav}.'
    qt_s = min(40, qtrs) / 40 * 40; al_s = 20 if alpha > 0 else 0
    sh_s = 15 if sh_p > sh_b else 0; dd_s = 15 if mdd > -50 else (8 if mdd > -65 else 0)
    try:
        rec_s = 10 if (datetime.now()-datetime.strptime(str(metrics.get('data_to','')), '%Y-%m-%d')).days < 730 else 0
    except: rec_s = 0
    return {'signal': signal, 'interpretation': interpretation,
            'reason': reason, 'confidence_score': int(qt_s + al_s + sh_s + dd_s + rec_s)}


# ── Column priority order (uses {bm}_ placeholder) ────────────────────────

PRIORITY_COLS_TEMPLATE = [
    'type','name','links',
    'signal','confidence_score','score_vs_benchmark','good_overall',
    'is_outperforming_{bm}','interpretation','reason',
    'good_return','good_total_return','good_xirr',
    'good_rolling_1y','good_rolling_3y',
    'good_win_rate','good_avg_return',
    'good_sharpe','good_sortino','good_calmar',
    'good_volatility','good_downside_vol',
    'good_drawdown','good_peak_to_curr','good_consec_losses',
    '{bm}_ann_return_pct',        'ann_return_pct',
    '{bm}_total_return_pct',      'total_return_pct',
    '{bm}_xirr_approx_pct',       'xirr_approx_pct',
    '{bm}_rolling_1y_pct',        'rolling_1y_pct',
    '{bm}_rolling_3y_pct',        'rolling_3y_pct',
    '{bm}_win_rate_pct',          'win_rate_pct',
    '{bm}_avg_qtr_return_pct',    'avg_qtr_return_pct',
    '{bm}_median_qtr_return_pct', 'median_qtr_return_pct',
    '{bm}_volatility_ann_pct',    'volatility_ann_pct',
    '{bm}_downside_vol_ann_pct',  'downside_vol_ann_pct',
    '{bm}_sharpe_ratio',          'sharpe_ratio',
    '{bm}_sortino_ratio',         'sortino_ratio',
    '{bm}_calmar_ratio',          'calmar_ratio',
    '{bm}_max_drawdown_pct',      'max_drawdown_pct',
    '{bm}_peak_to_current_pct',   'peak_to_current_pct',
    '{bm}_max_consec_losses',     'max_consec_losses',
    '{bm}_skew',                  'skew',
    '{bm}_kurtosis',              'kurtosis',
    'alpha_ann_pct','beta','correlation_to_bm',
    'tracking_error_pct','information_ratio',
    'current_net_worth_cr','data_from','data_to','quarters_tracked','portfolio_url',
]

def get_priority_cols(bm):
    return [c.replace('{bm}', bm) for c in PRIORITY_COLS_TEMPLATE]


# ── Monthly history snapshot ──────────────────────────────────────────────

# 13 key columns stored per investor per month-start run
HISTORY_COLS_TEMPLATE = [
    'type', 'name', 'signal', 'confidence_score', 'score_vs_benchmark',
    'good_overall', 'alpha_ann_pct', 'sharpe_ratio', 'ann_return_pct',
    'rolling_1y_pct', 'max_drawdown_pct', 'peak_to_current_pct',
    'is_outperforming_{bm}',
]

def append_history(sheet_key, history_tab, clean_df, bm):
    """
    Append one row per investor to the history tab — never overwrites, only appends.
    Creates the tab + header row automatically on the first call.
    Called only when today is the 1st of the month.
    """
    today = datetime.now().strftime('%Y-%m-%d')
    cols  = [c.replace('{bm}', bm) for c in HISTORY_COLS_TEMPLATE]
    avail = [c for c in cols if c in clean_df.columns]

    hist_df = clean_df[avail].copy()
    hist_df.insert(0, 'run_date', today)
    header = hist_df.columns.tolist()
    rows   = hist_df.fillna('').values.tolist()

    try:
        book = _gsheet_client().open_by_key(sheet_key)
        try:
            ws = book.worksheet(history_tab)
            if not ws.get_all_values():          # tab exists but empty
                ws.append_row(header)
        except gspread.exceptions.WorksheetNotFound:
            ws = book.add_worksheet(title=history_tab, rows=10000, cols=len(header) + 2)
            ws.append_row(header)
        ws.append_rows(rows)
        print(f'  History: appended {len(rows)} rows to "{history_tab}" (run_date={today}).')
    except Exception as e:
        print(f'  append_history error: {e}')


# ── Generic metrics loop + write ──────────────────────────────────────────

# Block detection (mirrors MASTER_MIN_SUCCESS / the consecutive-fail cooldown in build_master_stock):
# judge the run on FETCH success (fraction of investors whose page loaded, incl. legit chart-less
# 'empty'/short-history), NOT on metric completeness. Below this fraction the run is treated as
# blocked and REFUSES to overwrite the summary tab.
METRICS_MIN_SUCCESS   = 0.40
METRICS_MAX_CONSEC_FAIL = 10   # this many consecutive fetch FAILS -> site-wide block -> stop early


def run_metrics_loop(investors_df, benchmark_series, risk_free_rate, bm,
                     summary_tab, links_tab, checkpoint_file=None):
    """
    Runs the full metrics pipeline for any investor list. Checkpoints after each investor.
    On the 1st of each month, also appends a snapshot to the history tab.

    Block-safe (mirrors build_master_stock): every investor is tagged _status ok/empty/short/fail;
    only a real fetch failure ('fail', after retries) counts as a block. On resume, 'fail' rows are
    DROPPED so they are retried. A burst of consecutive fails trips a circuit-breaker (fail-fast on a
    site-wide block instead of grinding through retries x N). The write is judged on FETCH success,
    not metric completeness, and a PARTIAL block never erases a blocked investor's last-good row -
    those rows are merged back from the existing tab.
    """
    if checkpoint_file is None:
        checkpoint_file = os.path.join(os.getcwd(), f'{summary_tab}_checkpoint.csv')

    done_names = set()
    rows = []
    if os.path.exists(checkpoint_file):
        try:
            ckpt = pd.read_csv(checkpoint_file)
            rows = ckpt.where(ckpt.notna(), None).to_dict('records')
            rows = [r for r in rows if str(r.get('_status')) != 'fail']        # drop fails -> retry them
            done_names = {str(r.get('name', '')).strip() for r in rows if str(r.get('name', '')).strip()}
            print(f'Resuming from checkpoint: {len(done_names)} done; previous failures will be RETRIED.')
        except Exception as e:
            print(f'Checkpoint load failed ({e}), starting fresh.'); rows, done_names = [], set()

    total = len(investors_df)
    print(f'Processing {total} investors (~{REQUEST_DELAY*(total-len(done_names))/60:.0f} min remaining)...\n')

    # Warmed browser session (cookies + full headers) on the CORRECT domain -> far fewer blocks on
    # long / CI runs. India lives on trendlyne.com, US on us.trendlyne.com.
    _base = 'https://us.trendlyne.com' if bm == 'sp500' else 'https://trendlyne.com'
    sess = _make_session(_base)
    try:
        time.sleep(REQUEST_DELAY); sess.get(_base, timeout=20); time.sleep(REQUEST_DELAY)
    except Exception:
        pass

    consec_fail = 0
    aborted_block = False
    try:
        for i, row in investors_df.iterrows():
            name     = str(row.get('name','') or '').strip()
            inv_type = str(row.get('type','') or '').strip()
            url      = str(row.get('links','') or row.get('portfolio_url','')).strip()
            if not url or url == 'nan': continue

            if name in done_names:
                print(f'[{i+1:>3}/{total}] {name[:40]:<40} (cached)')
                continue

            print(f'[{i+1:>3}/{total}] {name[:40]:<40}', end=' ')
            pf, fstatus = fetch_portfolio_history(url, session=sess)
            base = {'type': inv_type, 'name': name, 'links': url}

            if fstatus == 'fail':
                consec_fail += 1
                print('FAIL (fetch/blocked)')
                base.update({'_status':'fail','signal':'','confidence_score':0,f'is_outperforming_{bm}':''})
                rows.append(base)
                if consec_fail >= METRICS_MAX_CONSEC_FAIL:
                    print(f'\n⏹  {consec_fail} consecutive fetch failures - treating as a site-wide block '
                          f'and stopping early (no partial overwrite).')
                    aborted_block = True; break
            elif pf.empty or len(pf) < MIN_QUARTERS:
                consec_fail = 0
                print(f'skip ({len(pf) if not pf.empty else 0} qtrs)')
                base.update({'_status':'short','signal':'','confidence_score':0,'score_vs_benchmark':'',
                             'good_overall':'',f'is_outperforming_{bm}':'',
                             'quarters_tracked': len(pf) if not pf.empty else 0})
                rows.append(base)
            else:
                consec_fail = 0
                metrics = compute_metrics(pf, benchmark_series, risk_free_rate, bm)
                if not metrics:
                    print('metrics failed')
                    base.update({'_status':'short','signal':'','confidence_score':0,f'is_outperforming_{bm}':''})
                    rows.append(base)
                else:
                    interp = make_interpretation_row(metrics, bm)
                    flags  = compute_comparison_flags(metrics, bm)
                    base.update({'_status':'ok'}); base.update(metrics); base.update(interp); base.update(flags)
                    rows.append(base)
                    print(f"sig={interp['signal']:<10} conf={interp['confidence_score']:>3}  "
                          f"alpha={str(metrics.get('alpha_ann_pct','?')):>6}%  "
                          f"sh={str(metrics.get('sharpe_ratio','?')):>5}  "
                          f"mdd={str(metrics.get('max_drawdown_pct','?')):>7}%  "
                          f"{flags['score_vs_benchmark']}")

            done_names.add(name)
            try:
                pd.DataFrame(rows).to_csv(checkpoint_file, index=False)
            except Exception:
                pass

    except KeyboardInterrupt:
        print(f'\nInterrupted! {len(rows)} investors saved to checkpoint:')
        print(f'  {checkpoint_file}')
        print('Re-run this cell to resume (failures will be retried).')
        return

    # ── Build final DataFrame + fetch-outcome accounting ─────────────────────────
    summary_df = pd.DataFrame(rows)
    if summary_df.empty:
        print('No results.'); return

    _st = summary_df['_status'].astype(str) if '_status' in summary_df.columns else pd.Series('ok', index=summary_df.index)
    attempted  = len(summary_df)
    n_fail     = int((_st == 'fail').sum())
    fetch_succ = (attempted - n_fail) / max(1, attempted)

    def _has_metric(v):
        return str(v).strip().lower() not in ('', 'nan', 'none')
    mask_full = summary_df['ann_return_pct'].apply(_has_metric) if 'ann_return_pct' in summary_df.columns \
                else pd.Series(False, index=summary_df.index)
    n_full = int(mask_full.sum())
    print(f'\nFetch: {attempted - n_fail}/{attempted} ok ({fetch_succ:.0%}) - investors with full metrics: {n_full}')

    # ── Block guard (FETCH success) - preserve the tab, keep fails for retry ─────
    if aborted_block or n_full == 0 or fetch_succ < METRICS_MIN_SUCCESS:
        print(f'\n\U0001f6a8 ALERT: fetch success {fetch_succ:.0%} ({n_fail}/{attempted} failed), full metrics '
              f'{n_full} - Trendlyne likely blocked this IP. "{summary_tab}" LEFT UNCHANGED (existing data '
              f'preserved). Failed investors stay in the checkpoint and are RETRIED next run.')
        return

    # ── This run's full-metric rows, sorted best-first ───────────────────────────
    good = summary_df[mask_full].copy()
    col_order = [c for c in get_priority_cols(bm) if c in good.columns]
    remaining = [c for c in good.columns if c not in col_order and c != '_status']
    good = good[col_order + remaining]
    sig_rank = {'STRONG BUY':0,'BUY':1,'WATCH':2,'HOLD':3,'AVOID':4,'':5}
    def _scol(df, _n, _d):
        return df[_n] if _n in df.columns else pd.Series(_d, index=df.index)
    good['_s'] = _scol(good,'signal','').map(sig_rank).fillna(5)
    good['_c'] = pd.to_numeric(_scol(good,'confidence_score',0), errors='coerce').fillna(0)
    good['_a'] = pd.to_numeric(_scol(good,'alpha_ann_pct',''),   errors='coerce').fillna(-999)
    good = good.sort_values(['_s','_c','_a'], ascending=[True,False,False]) \
               .drop(columns=['_s','_c','_a']).reset_index(drop=True)

    # ── PARTIAL-block protection: for investors ATTEMPTED but not freshly 'ok' (blocked/short),
    #    keep their last-good row from the existing tab instead of dropping or zeroing them. ──
    clean_df = good
    try:
        prior = read_sheet(SHEET_KEY, summary_tab)
    except Exception:
        prior = pd.DataFrame()
    if not prior.empty and 'name' in prior.columns:
        prior = prior[prior['name'].astype(str).str.strip() != '']
        attempted_names = set(summary_df['name'].astype(str).str.strip())
        have_now        = set(good['name'].astype(str).str.strip())
        keep = prior[prior['name'].astype(str).str.strip().isin(attempted_names - have_now)]
        if not keep.empty:
            clean_df = pd.concat([good, keep], ignore_index=True, sort=False)
            print(f'  Merged {len(keep)} last-good row(s) for investors not freshly scraped this run.')
    clean_df = clean_df[[c for c in clean_df.columns if c != '_status']]

    dropped = attempted - len(clean_df)
    print(f'Writing {len(clean_df)} rows to "{summary_tab}" (dropped {dropped} skip/short rows).')
    if 'signal' in clean_df.columns:
        print(clean_df['signal'].value_counts().to_string())

    # ── Write snapshot ────────────────────────────────────────────────────
    write_sheet(SHEET_KEY, summary_tab, clean_df)

    # ── Monthly history append (1st of month only) ────────────────────────
    today = datetime.now()
    if today.day == 1:
        history_tab = summary_tab.replace('_summary', '_history')
        print(f'\n1st of month ({today.strftime("%Y-%m-%d")}) - appending history snapshot...')
        append_history(SHEET_KEY, history_tab, clean_df, bm)
    else:
        print(f'\nHistory skipped - runs on 1st of month (today is {today.strftime("%d %b")})')

    # ── Preview ───────────────────────────────────────────────────────────
    preview = [c for c in ['type','name','signal','confidence_score','score_vs_benchmark',
                             'alpha_ann_pct','sharpe_ratio','max_drawdown_pct',
                             'ann_return_pct','good_overall'] if c in clean_df.columns]
    print(f'\n-- Top 15 by Signal + Confidence --')
    print(clean_df[preview].head(15).to_string(index=False))

    try:
        if os.path.exists(checkpoint_file):
            os.remove(checkpoint_file)
            print(f'\nCheckpoint deleted (run complete).')
    except Exception:
        pass


print('Metrics engine ready.')

Metrics engine ready.


---
## INDIA — Nifty 50 Benchmark  |  RFR = 6.5%

Steps: **Cell 6** (scrape) → **Cell 7** (Nifty data) → **Cell 8** (metrics)
> Cell 6 writes `type | name | links` to `indian_fii_dii_trend_lyne_links`.
> Cell 8 writes full metrics to `fii_dii_indian_investment_summary` (empty rows dropped).


In [6]:
# ── CELL 6: SCRAPE INDIAN INVESTORS ──────────────────────────────────────
ind_df = scrape_india('https://trendlyne.com/portfolio/superstar-shareholders/index/',               'individual investor')
ins_df = scrape_india('https://trendlyne.com/portfolio/superstar-shareholders/index/institutional/', 'institutional investor')
fii_df = scrape_india('https://trendlyne.com/portfolio/superstar-shareholders/index/fii/',            'fii')

frames = [f for f in [ind_df, ins_df, fii_df] if not f.empty]
if not frames:
    print('ERROR: All Indian scrapes failed.')
else:
    all_india = pd.concat(frames, ignore_index=True)
    print(f'\nTotal Indian investors: {len(all_india)}')
    print(all_india['type'].value_counts().to_string())
    write_sheet(SHEET_KEY, 'indian_fii_dii_trend_lyne_links', all_india)
    display(all_india.head(8))

  Scraping individual investor ...
  Found 62 investors.
  Scraping institutional investor ...
  Found 124 investors.
  Scraping fii ...
  Found 68 investors.

Total Indian investors: 254
type
institutional investor    124
fii                        68
individual investor        62
  Wrote 254 rows to "indian_fii_dii_trend_lyne_links".


,type,name,links
0,individual investor,ajay upadhyaya,https://trendlyne.com/portfolio/superstar-shar...
1,individual investor,akash bhanshali,https://trendlyne.com/portfolio/superstar-shar...
2,individual investor,amit gupta,https://trendlyne.com/portfolio/superstar-shar...
3,individual investor,anil kumar goel and associates,https://trendlyne.com/portfolio/superstar-shar...
4,individual investor,anuj anantrai sheth and associates,https://trendlyne.com/portfolio/superstar-shar...
5,individual investor,ashish dhawan,https://trendlyne.com/portfolio/superstar-shar...
6,individual investor,ashish kacholia,https://trendlyne.com/portfolio/superstar-shar...
7,individual investor,ashok kumar jain,https://trendlyne.com/portfolio/superstar-shar...


In [7]:
# ── CELL 7: DOWNLOAD NIFTY 50 ────────────────────────────────────────────
print('Downloading Nifty 50...')
nifty_series = yf.Ticker(INDIA_BENCHMARK).history(start='2005-01-01', auto_adjust=True)['Close']
if nifty_series.index.tz is not None:
    nifty_series.index = nifty_series.index.tz_localize(None)
nifty_series.index = pd.to_datetime(nifty_series.index)
print(f'Nifty 50: {nifty_series.index.min().date()} -> {nifty_series.index.max().date()} ({len(nifty_series):,} days)')

_nr = nifty_series.pct_change().dropna()
_nq = _nr.resample('QE').apply(lambda x:(1+x).prod()-1).dropna().values
_an = (np.prod(1+_nq)**(4/len(_nq))-1)*100; _vn = np.std(_nq,ddof=1)*np.sqrt(4)*100
print(f'Nifty reference: ann_return={_an:.1f}%  vol={_vn:.1f}%  sharpe={(_an/100-INDIA_RFR)/(_vn/100):.3f}  rfr={INDIA_RFR*100:.1f}%')

Nifty 50: 2007-09-17 -> 2026-07-07 (4,611 days)
Nifty reference: ann_return=9.2%  vol=20.9%  sharpe=0.129  rfr=6.5%


In [8]:
# ── CELL 8: INDIAN METRICS + WRITE ───────────────────────────────────────
india_df = read_sheet(SHEET_KEY, 'indian_fii_dii_trend_lyne_links')
if india_df.empty:
    print('No Indian investors. Run Cell 6 first.')
else:
    run_metrics_loop(
        investors_df     = india_df,
        benchmark_series = nifty_series,
        risk_free_rate   = INDIA_RFR,
        bm               = 'nifty',
        summary_tab      = 'fii_dii_indian_investment_summary',
        links_tab        = 'indian_fii_dii_trend_lyne_links',
    )

Processing 254 investors (~6 min remaining)...

[  1/254] ajay upadhyaya                           sig=WATCH      conf= 85  alpha= 44.04%  sh=0.324  mdd= -77.01%  10/15
[  2/254] akash bhanshali                          sig=WATCH      conf= 85  alpha= 53.03%  sh=0.344  mdd= -79.76%  10/15
[  3/254] amit gupta                               sig=AVOID      conf= 50  alpha=-16.92%  sh=-0.288  mdd=  -88.6%  0/15
[  4/254] anil kumar goel and associates           sig=HOLD       conf= 70  alpha=  4.74%  sh=0.164  mdd=  -73.2%  6/15
[  5/254] anuj anantrai sheth and associates       sig=AVOID      conf= 50  alpha= -1.06%  sh=0.029  mdd= -83.13%  1/15
[  6/254] ashish dhawan                            sig=HOLD       conf= 70  alpha=  4.93%  sh=0.116  mdd= -67.64%  4/15
[  7/254] ashish kacholia                          sig=BUY        conf= 93  alpha= 12.03%  sh=0.339  mdd= -55.28%  8/15
[  8/254] ashok kumar jain                         sig=AVOID      conf= 70  alpha=  5.55%  sh=0.094  mdd= -83

In [9]:
# ── CELL 8b: DETECT PER-INVESTOR QUARTER UPDATES ─────────────────────────────
# Diffs each Indian investor's `data_to` (their latest disclosed quarter) against the
# PREVIOUS run's stored value and logs the advances to `quarter_updates`, so the dashboard
# can show "Mukul Agrawal: 2026-03-31 -> 2026-06-30". Last-seen values live in `quarter_state`.
# Best-effort: any error is printed and swallowed so it never breaks the daily run.
try:
    _sum = read_sheet(SHEET_KEY, 'fii_dii_indian_investment_summary')
    if not _sum.empty and {'name', 'data_to'}.issubset(_sum.columns):
        _new = {str(n).strip(): str(d).strip()
                for n, d in zip(_sum['name'], _sum['data_to'])
                if str(n).strip() and str(d).strip() and str(d).strip().lower() != 'nan'}
        _state = read_sheet(SHEET_KEY, 'quarter_state')
        _old = ({str(n).strip(): str(d).strip() for n, d in zip(_state['name'], _state['data_to'])}
                if (not _state.empty and {'name', 'data_to'}.issubset(_state.columns)) else {})
        _today = datetime.now().strftime('%Y-%m-%d')
        # ISO yyyy-mm-dd compares correctly as a string, so q > _old[n] == "advanced to a newer quarter"
        _changes = [{'detected_on': _today, 'name': n, 'prev_quarter': _old.get(n, ''), 'new_quarter': q}
                    for n, q in _new.items() if n in _old and _old[n] and q > _old[n]]
        if _changes:
            _upd = pd.DataFrame(_changes)
            _prev = read_sheet(SHEET_KEY, 'quarter_updates')
            if not _prev.empty:
                _upd = pd.concat([_prev, _upd], ignore_index=True)
            write_sheet(SHEET_KEY, 'quarter_updates', _upd)
            print(f'  quarter_updates: {len(_changes)} portfolio(s) advanced to a newer quarter.')
        else:
            print('  quarter_updates: no portfolios advanced this run' + (' (baseline set).' if not _old else '.'))
        write_sheet(SHEET_KEY, 'quarter_state',
                    pd.DataFrame([{'name': n, 'data_to': q} for n, q in _new.items()]))
    else:
        print('  quarter-update check skipped - summary missing name/data_to.')
except Exception as _e:
    print(f'  quarter-update check error (non-fatal): {_e}')


  quarter_updates: no portfolios advanced this run.
  Wrote 186 rows to "quarter_state".


---
## US — S&P 500 Benchmark  |  RFR = 4.5%

Steps: **Cell 9** (scrape) → **Cell 10** (S&P 500 data) → **Cell 11** (metrics)
> Cell 9 writes `type | name | links` to `us_fii_dii_trend_lyne_links`.
> Cell 11 writes full metrics to `fii_dii_us_investment_summary` (empty rows dropped).


In [10]:
# ── CELL 9: SCRAPE US INVESTORS ──────────────────────────────────────────
# Use a warmed-up session so us.trendlyne.com does not return 405
_us_session = _make_session('https://us.trendlyne.com')

us_ind_df = scrape_us('https://us.trendlyne.com/us/portfolio/superstar-shareholders/index/',               'individual investor',   _us_session)
us_ins_df = scrape_us('https://us.trendlyne.com/us/portfolio/superstar-shareholders/index/institutional/', 'institutional investor', _us_session)

frames = [f for f in [us_ind_df, us_ins_df] if not f.empty]
if not frames:
    print('ERROR: All US scrapes failed.')
else:
    all_us = pd.concat(frames, ignore_index=True)
    print(f'\nTotal US investors: {len(all_us)}')
    print(all_us['type'].value_counts().to_string())
    write_sheet(SHEET_KEY, 'us_fii_dii_trend_lyne_links', all_us)
    display(all_us.head(8))

  Scraping individual investor ...
  ERROR: 405 Client Error: Not Allowed for url: https://us.trendlyne.com/us/portfolio/superstar-shareholders/index/
  Scraping institutional investor ...
  ERROR: 405 Client Error: Not Allowed for url: https://us.trendlyne.com/us/portfolio/superstar-shareholders/index/institutional/
ERROR: All US scrapes failed.


In [11]:
# ── CELL 10: DOWNLOAD S&P 500 ────────────────────────────────────────────
print('Downloading S&P 500...')
sp500_series = yf.Ticker(US_BENCHMARK).history(start='2005-01-01', auto_adjust=True)['Close']
if sp500_series.index.tz is not None:
    sp500_series.index = sp500_series.index.tz_localize(None)
sp500_series.index = pd.to_datetime(sp500_series.index)
print(f'S&P 500: {sp500_series.index.min().date()} -> {sp500_series.index.max().date()} ({len(sp500_series):,} days)')

_sr = sp500_series.pct_change().dropna()
_sq = _sr.resample('QE').apply(lambda x:(1+x).prod()-1).dropna().values
_as = (np.prod(1+_sq)**(4/len(_sq))-1)*100; _vs = np.std(_sq,ddof=1)*np.sqrt(4)*100
print(f'S&P 500 reference: ann_return={_as:.1f}%  vol={_vs:.1f}%  sharpe={(_as/100-US_RFR)/(_vs/100):.3f}  rfr={US_RFR*100:.1f}%')

S&P 500: 2005-01-03 -> 2026-07-07 (5,410 days)
S&P 500 reference: ann_return=8.8%  vol=15.7%  sharpe=0.272  rfr=4.5%


In [12]:
# ── CELL 11: US METRICS + WRITE ──────────────────────────────────────────
us_df = read_sheet(SHEET_KEY, 'us_fii_dii_trend_lyne_links')
if us_df.empty:
    print('No US investors. Run Cell 9 first.')
else:
    run_metrics_loop(
        investors_df     = us_df,
        benchmark_series = sp500_series,
        risk_free_rate   = US_RFR,
        bm               = 'sp500',
        summary_tab      = 'fii_dii_us_investment_summary',
        links_tab        = 'us_fii_dii_trend_lyne_links',
    )

Processing 87 investors (~2 min remaining)...

[  1/87] warren buffett                               fetch failed after 4 tries (ders/2107312/latest/warren-buffett-portfolio/): HTTP 405
FAIL (fetch/blocked)
[  2/87] ken fisher                                   fetch failed after 4 tries (eholders/2158574/latest/ken-fisher-portfolio/): HTTP 405
FAIL (fetch/blocked)
[  3/87] baillie gifford                              fetch failed after 4 tries (ers/2158576/latest/baillie-gifford-portfolio/): HTTP 405
FAIL (fetch/blocked)
[  4/87] kenneth c. griffin                           fetch failed after 4 tries (s/2158577/latest/kenneth-c-griffin-portfolio/): HTTP 405
FAIL (fetch/blocked)
[  5/87] jim simons                               sig=AVOID      conf= 78  alpha=  4.78%  sh=-0.005  mdd= -20.13%  1/15
[  6/87] bill gates                               sig=AVOID      conf= 78  alpha=  0.42%  sh=-0.487  mdd= -32.89%  0/15
[  7/87] ronald s. baron                          sig=BUY        conf= 93

---
## Metric Glossary
Run **Cell 12** once to write the full metric definitions to `fii_dii_metric_docs`.


In [13]:
# ── CELL 12: WRITE METRIC GLOSSARY ───────────────────────────────────────
GLOSSARY_COLS = ['metric','category','plain_english','formula',
                 'good_value','bad_value','compare_with_benchmark','decision_use']

GLOSSARY = [
    ('ann_return_pct','Returns','Geometric annualised return over the full data window',
     '(∏(1+quarterly_return_i))^(4/n) − 1 × 100',
     '> benchmark ann return; sustained > 15% for India, > 12% for US',
     '< 0% annual; < benchmark',
     'YES — direct comparison','Primary filter. If < benchmark, check alpha and Sharpe before following'),
    ('total_return_pct','Returns','Total % gain from first to last quarter, not annualised',
     '(last_nw / first_nw − 1) × 100',
     '> benchmark total return over same window',
     '< 0% = net loss over full history',
     'YES — same window as investor',
     'Use alongside ann_return to spot front-loaded vs recent performance'),
    ('xirr_approx_pct','Returns','Annualised CAGR — treats first net-worth as lump-sum. True XIRR needs actual cash-flow dates',
     '(last_nw / first_nw)^(1/years) − 1 × 100',
     '> benchmark CAGR; > 15% India, > 12% US',
     '< 8% India / < 5% US = worse than risk-free alternatives',
     'YES','Most intuitive headline return metric'),
    ('rolling_1y_pct','Returns','Return over most recent 4 quarters — current momentum',
     '(∏(1+last_4_qtrs)) − 1 × 100',
     '> 0%; > benchmark rolling 1Y',
     '< −15% = investor currently in trouble',
     'YES','Check before following recent trades. Negative = currently struggling'),
    ('rolling_3y_pct','Returns','Return over most recent 12 quarters — medium-term trend',
     '(∏(1+last_12_qtrs)) − 1 × 100',
     '> benchmark rolling 3Y; > 50% India, > 35% US cumulative',
     '< 0% over 3 years = sustained underperformance',
     'YES','Best horizon for skill evaluation. Smooths single-year flukes'),
    ('win_rate_pct','Consistency','% of quarters with positive return',
     'count(r>0) / n × 100',
     '> 60%; > benchmark win rate',
     '< 45% = losing money more than half the time',
     'YES','High win rate + high return = true skill'),
    ('avg_qtr_return_pct','Consistency','Arithmetic mean of quarterly returns. Sensitive to outliers',
     'mean(quarterly_returns) × 100',
     '> benchmark avg quarterly return',
     '< 0%',
     'YES','If avg >> median, a few huge quarters inflate it. Check skew'),
    ('median_qtr_return_pct','Consistency','Median quarterly return. Outlier-resistant',
     'median(quarterly_returns) × 100',
     '> benchmark median quarterly return',
     '< 0% = more than half of quarters are negative',
     'YES','More reliable than avg for typical performance'),
    ('volatility_ann_pct','Risk','Annualised standard deviation of quarterly returns',
     'std(quarterly_returns, ddof=1) × √4 × 100',
     '< benchmark vol; < 25% annualised acceptable',
     '> 45% = extremely volatile',
     'YES — lower is better','Never evaluate alone. High vol + high return may be fine if Sharpe is strong'),
    ('downside_vol_ann_pct','Risk','Std dev of NEGATIVE returns only, annualised. Pure downside risk',
     'std(negative_quarterly_returns, ddof=1) × √4 × 100',
     '< benchmark downside vol; < 20%',
     '> 35% = heavy losses when things go wrong',
     'YES — lower is better','Used in Sortino ratio. Low = investor protects capital in bad markets'),
    ('sharpe_ratio','Risk-Adjusted','Excess return above risk-free per unit of TOTAL volatility. Primary risk-adjusted metric',
     '(ann_return/100 − rfr) / ann_vol   [India rfr=6.5%, US rfr=4.5%]',
     '> 1.0 = excellent; > benchmark Sharpe',
     '< 0 = return below risk-free rate. FD beats this portfolio',
     'YES — MOST IMPORTANT COMPARISON','Only follow investors whose Sharpe exceeds benchmark Sharpe. Raw return alone is never enough'),
    ('sortino_ratio','Risk-Adjusted','Like Sharpe but only penalises DOWNSIDE volatility. Fairer for buy-and-hold strategies',
     '(ann_return/100 − rfr) / downside_vol',
     '> 1.5 = excellent; > benchmark Sortino',
     '< 0 = return below risk-free even adjusted for only downside risk',
     'YES','Preferred over Sharpe for concentrated value investors'),
    ('calmar_ratio','Risk-Adjusted','Annual return divided by worst drawdown ever. How much earned per crash risk?',
     'ann_return (decimal) / |max_drawdown (decimal)|',
     '> 0.5 good; > 1.0 excellent; > benchmark Calmar',
     '< 0.2 = very low reward for crash risk taken',
     'YES','Best for risk-averse investors or those near retirement'),
    ('max_drawdown_pct','Risk','Worst peak-to-trough portfolio decline EVER. Example: −50% = halved at worst point',
     'min((value − rolling_peak) / rolling_peak × 100)  [most negative value]',
     '> −40% (less negative = smaller crash = better); > benchmark MDD',
     '< −70% = catastrophic drawdown at some point in history',
     'YES — less negative is better','Can you hold through a 60% drawdown? If not, do not follow this investor'),
    ('peak_to_current_pct','Risk','How far portfolio is RIGHT NOW from its all-time high. 0% = at ATH today',
     '(current_nw − ATH_nw) / ATH_nw × 100',
     '> −20%; > benchmark peak-to-current',
     '< −50% = portfolio in deep drawdown now. Risky entry',
     'YES — less negative = closer to ATH = better','0% at ATH = strong recovery. Deep negative = wait or avoid'),
    ('max_consec_losses','Risk','Longest streak of consecutive losing quarters in history',
     'Maximum run length of (quarterly_return < 0)',
     '<= 2 quarters; <= benchmark',
     '>= 5 = lost for over a year without recovery at some point',
     'YES — lower is better','Long streaks test conviction. 6+ consecutive losses questions the strategy'),
    ('skew','Tail Risk','Return distribution asymmetry. Positive = more big UP quarters than DOWN',
     'scipy.stats.skew(quarterly_returns)',
     '> 0 (positive skew = good surprises dominate)',
     '< −1.0 = high crash risk; occasional huge losses pull average down',
     'YES — positive skew vs benchmark preferred','Negative skew looks fine until the crash quarter. Always check MDD alongside'),
    ('kurtosis','Tail Risk','Fat-tail risk. High = extreme events happen more than normal distribution predicts',
     'scipy.stats.kurtosis(quarterly_returns)  [excess kurtosis; normal=0]',
     'Near 0 or slightly negative; < 2',
     '> 5 = heavy tails; returns have unpredictable extreme outliers',
     'YES — lower is better','High kurtosis + negative skew = danger zone'),
    ('alpha_ann_pct','Active','THE skill metric. Pure excess return vs benchmark AFTER removing market beta contribution',
     'CAPM: r_portfolio = alpha + beta × r_benchmark. Alpha = intercept × 4 × 100',
     '> 0% = skill; > 5% = strong alpha generator',
     '< 0% = Nifty/S&P ETF beats this investor on risk-adjusted basis',
     'N/A — alpha IS already the benchmark-neutral comparison','PRIMARY driver. Positive alpha = follow. Negative alpha = do not copy trades'),
    ('beta','Active','Portfolio amplification of benchmark moves. Beta 1.5 = moves 1.5% per 1% benchmark move',
     'Slope from OLS regression of portfolio returns on benchmark returns',
     '0.8–1.2 = market-like; < 0.8 = defensive',
     '> 2.0 = extremely aggressive; crash risk 2× the benchmark',
     'YES — low beta + high return = skill; high beta + high return = just leveraged the index','High beta needs much higher alpha to justify following'),
    ('correlation_to_bm','Active','Pearson correlation of portfolio with benchmark. 1.0 = lockstep',
     'corrcoef(portfolio_returns, benchmark_returns)',
     '< 0.7 = genuinely independent picks',
     '> 0.95 = closet index fund',
     'N/A — absolute; lower is better for active management','High correlation + no alpha = expensive index fund. Avoid'),
    ('tracking_error_pct','Active','How far portfolio deviates from benchmark path each year',
     'std(portfolio_returns − benchmark_returns, ddof=1) × √4 × 100',
     '> 10% = genuinely active',
     '< 5% = hugging benchmark; active in name only',
     'N/A — must pair with information_ratio','High TE + high IR = skilled. High TE + low IR = reckless'),
    ('information_ratio','Active','Active return per unit of active risk. Consistency of outperformance',
     'mean(r_port − r_bm) / std(r_port − r_bm) × √4',
     '> 0.5 = good; > 1.0 = top-tier',
     '< 0 = active bets destroy value',
     'N/A — already a benchmark comparison','Definitive skill metric. IR > 0.5 over 5+ years is rare and valuable'),
    ('is_outperforming_nifty / is_outperforming_sp500','Active',
     'YES if investor beats benchmark on BOTH alpha (skill) AND Sharpe (risk efficiency)',
     'YES if alpha_ann_pct > 0 AND sharpe_ratio > benchmark_sharpe_ratio',
     'YES','NO = no reason to prefer over an index fund',
     'N/A — result of benchmark comparison','First filter. If NO, understand why. Only MDD issue = reconsider. Negative alpha = skip'),
    ('good_return','Flags','YES if ann_return_pct > benchmark ann return','ann_return > bm_ann_return','YES','NO','Direct','Basic return filter'),
    ('good_total_return','Flags','YES if total_return_pct > benchmark total return','total > bm_total','YES','NO','Direct','Full-history filter'),
    ('good_xirr','Flags','YES if xirr_approx_pct > benchmark xirr','xirr > bm_xirr','YES','NO','Direct','CAGR filter'),
    ('good_rolling_1y','Flags','YES if rolling_1y_pct > benchmark rolling 1Y','rolling_1y > bm_rolling_1y','YES','NO','Direct','Recent momentum filter'),
    ('good_rolling_3y','Flags','YES if rolling_3y_pct > benchmark rolling 3Y','rolling_3y > bm_rolling_3y','YES','NO','Direct','Medium-term filter'),
    ('good_win_rate','Flags','YES if win_rate_pct > benchmark win rate','win_rate > bm_win_rate','YES','NO','Direct','Consistency filter'),
    ('good_avg_return','Flags','YES if avg_qtr_return_pct > benchmark average quarter','avg > bm_avg','YES','NO','Direct','Average quarter filter'),
    ('good_sharpe','Flags','YES if sharpe_ratio > benchmark Sharpe. MOST IMPORTANT FLAG','sharpe > bm_sharpe','YES','NO','Direct','Risk-efficiency filter. If NO, be very cautious'),
    ('good_sortino','Flags','YES if sortino_ratio > benchmark Sortino','sortino > bm_sortino','YES','NO','Direct','Capital protection filter'),
    ('good_calmar','Flags','YES if calmar_ratio > benchmark Calmar','calmar > bm_calmar','YES','NO','Direct','Crash-adjusted return filter'),
    ('good_volatility','Flags','YES if volatility_ann_pct < benchmark vol (lower = better)','vol < bm_vol','YES','NO','Direct — lower is better','Smooth ride filter'),
    ('good_downside_vol','Flags','YES if downside_vol_ann_pct < benchmark downside vol (lower = better)','dvol < bm_dvol','YES','NO','Direct — lower is better','Loss severity filter'),
    ('good_drawdown','Flags','YES if max_drawdown_pct > benchmark MDD (less negative = smaller crash = better)','mdd > bm_mdd numerically e.g. -30 > -50','YES','NO','Direct — less negative is better','Worst-case loss filter'),
    ('good_peak_to_curr','Flags','YES if peak_to_current_pct > benchmark (closer to ATH = better)','p2c > bm_p2c numerically','YES','NO','Direct — less negative is better','Current recovery filter'),
    ('good_consec_losses','Flags','YES if max_consec_losses < benchmark (fewer = better)','mcl < bm_mcl','YES','NO','Direct — lower is better','Losing streak filter'),
    ('score_vs_benchmark','Flags','Count of good_* flags that are YES. Format: X/Y e.g. 11/15',
     'sum(good_* == YES) / count(good_* in [YES,NO])',
     '>= 10/15 = strong across-the-board outperformer',
     '< 7/15 = beats benchmark on fewer than half the metrics',
     'N/A — summary of all flags','Combined filter: >= 10/15 + positive alpha = high-conviction'),
    ('good_overall','Flags','YES if investor beats benchmark on >= 60% of good_* flags',
     'YES if sum(YES) / total >= 0.60',
     'YES','NO = underperforms on more than 40% of dimensions',
     'N/A','Quick pre-filter. Not sufficient alone — check signal + confidence'),
    ('signal','Meta',
     'STRONG BUY / BUY / WATCH / HOLD / AVOID. Derived from alpha, Sharpe, Calmar, MDD',
     'STRONG BUY: alpha>5% AND sharpe>0.8 AND MDD>-40%. BUY: alpha>0 AND sharpe>bm AND calmar>0.3. WATCH: alpha>0 AND sharpe>bm. AVOID: alpha<0 OR sharpe<0 OR MDD<-75%. HOLD: rest. MDD<-70% downgrades BUY->WATCH',
     'STRONG BUY or BUY','AVOID',
     'N/A','STRONG BUY = follow closely. BUY = normal sizing. WATCH = monitor. AVOID = ignore or contrarian'),
    ('confidence_score','Meta',
     'Reliability of metrics. 0-100. Based on data length and quality flags',
     'data_length(max 40) + alpha_positive(20) + sharpe_beats_bm(15) + MDD_ok(15) + recency_2yrs(10)',
     '> 70 reliable; > 85 highly reliable',
     '< 40 = very short history, treat as indicative only',
     'N/A','Always check alongside signal. STRONG BUY with confidence 25 < BUY with confidence 85'),
    ('interpretation','Meta',
     'Auto-generated one-line plain-English summary of alpha quality, risk efficiency, drawdown, and data adequacy',
     'Rule-based text from alpha + Sharpe + MDD + sample size',
     'Clear positive statement','Contains: small sample, underperforms, poor risk/reward',
     'N/A','Read first before reviewing numbers. Instant context'),
    ('reason','Meta',
     'Top 2 drivers behind the signal — most impactful positive or negative factors',
     'Top 2 of: alpha value, Sharpe vs benchmark, MDD severity',
     'alpha +X% | Sharpe Y > BM Z','alpha -X% | MDD -Y%',
     'N/A','Explains WHY the signal was assigned. Use to validate or override'),
]

docs_df = pd.DataFrame(GLOSSARY, columns=GLOSSARY_COLS)
write_sheet(SHEET_KEY, 'fii_dii_metric_docs', docs_df)
print(f'Glossary written: {len(docs_df)} metric definitions across {docs_df["category"].nunique()} categories.')

  Wrote 45 rows to "fii_dii_metric_docs".
Glossary written: 45 metric definitions across 8 categories.


---
## Master Stock — superstar universe (daily)

**Cell 13** loads the helpers; **Cell 14** scrapes EVERY Indian investor's holdings (from `indian_fii_dii_trend_lyne_links`), unions them into one row per stock, and writes the **`master_stock`** tab — `ticker · company · superstar_count · held_by · total_value_cr · recent_action · as_of`.

Run **Cell 6** (scrape links) first. The build is **checkpointed** (safe to interrupt / re-run). If the scrape looks **broken** (site block / layout change — too few stocks or success < 60%), it **leaves `master_stock` untouched and prints an ALERT** — it never overwrites good data with a bad pull.

In [14]:
# ── CELL 13: MASTER STOCK — superstar universe (union of all holdings) ──
from urllib.parse import unquote

# Politeness / anti-block knobs for the holdings scrape (raise MASTER_DELAY if still blocked):
MASTER_DELAY     = 4.0     # base seconds between fetches (1.5 was too fast → got rate-limited)
MASTER_JITTER    = 2.0     # + up to this much RANDOM extra each request (less bot-like)
MASTER_MAX_DELAY = 15.0    # adaptive ceiling — the delay grows after each throttle stall

def _stake(x):
    x = (x or "").strip()
    if x in ("", "-", "—", "–"):
        return None
    m = re.search(r"-?\d+\.?\d*", x.replace(",", ""))
    return float(m.group()) if m else None

def _value_cr(x):
    m = re.search(r"-?\d+\.?\d*", (x or "").replace(",", ""))
    return float(m.group()) if m else None

def _symbol_from_href(href):
    m = re.search(r"/equity/(?:share-holding|share|stock)[^/]*/\d+/([^/]+)/", href or "")
    return unquote(m.group(1)).strip().upper() if m else ""

def fetch_holdings(url, session=None, retries=4, delay=None):
    """Scrape ONE investor's Trendlyne portfolio → (holdings_list, ok_flag).
    RETRIES WITH BACK-OFF on a failed/throttled fetch — Trendlyne rate-limits long runs, so a
    transient 429/5xx/timeout must not become a permanent FAIL. ok_flag=False only after all
    retries fail (real break); a genuinely empty portfolio returns ([], True)."""
    tb = None
    for attempt in range(retries):
        try:
            time.sleep((REQUEST_DELAY if delay is None else delay) + np.random.uniform(0, MASTER_JITTER))
            r = session.get(url, timeout=20) if session is not None \
                else requests.get(url, headers=REQUEST_HEADERS, timeout=20)
            if r.status_code == 200:
                _t = BeautifulSoup(r.text, "html.parser").find("table")
                if _t is not None:
                    tb = _t
                    break
            # non-200, OR a 200 with NO holdings table (a cookie/challenge/interstitial page — very
            # common on the FIRST request of a session) → back off and RETRY (the next hit is warm).
            time.sleep(6 * (attempt + 1))
        except Exception:
            time.sleep(6 * (attempt + 1))
    if tb is None:
        return [], False
    head_row = next((tr for tr in tb.find_all("tr") if tr.find("th")), None)
    head_ths = head_row.find_all("th") if head_row is not None else tb.find_all("th")
    quarters = []
    for th in head_ths:
        m = re.match(r"([A-Za-z]{3})\s+(\d{4})", th.get_text(" ", strip=True))
        if not m:
            continue
        lab = f"{m.group(1)} {m.group(2)}"
        if quarters and quarters[-1] == lab:
            continue
        if lab in quarters or len(quarters) >= 16:
            break
        quarters.append(lab)
    if not quarters:
        return [], False
    body = tb.find("tbody") or tb
    parsed = []
    for tr in body.find_all("tr"):
        if tr.find_parent("table") is not tb:
            continue
        tds = tr.find_all("td", recursive=False)
        cells = [td.get_text(" ", strip=True) for td in tds]
        if len(cells) < 5 + len(quarters) or not cells[1].strip():
            continue
        a = tds[1].find("a", href=True)
        ticker = _symbol_from_href(a["href"]) if a else ""
        series = [_stake(v) for v in cells[5:5 + len(quarters)]]
        parsed.append((cells, ticker, series))
    if not parsed:
        return [], True
    # ── Filing-Awaited guard (PER COMPANY) ────────────────────────────────────────────────
    # Shareholding patterns are filed per-company, so within one portfolio some stocks have filed the
    # latest quarter and some haven't. A blank LATEST cell = that company hasn't filed yet, NOT that
    # the investor exited. So per row we skip trailing-blank quarters and classify off the last FILED
    # quarter (a genuine exit shows as a filed 0). This kills the phantom-EXIT flood.
    out = []
    for cells, ticker, series in parsed:
        idx = [i for i, v in enumerate(series) if v is not None]
        if not idx:
            continue
        j = idx[0]                                    # most recent FILED quarter for this company
        latest = series[j]
        prev = next((series[k] for k in idx if k > j), None)
        if latest == 0 and prev:
            move, delta = "EXIT", -prev
        elif prev is None:
            move, delta = "NEW", latest
        else:
            d = round(latest - prev, 2)
            move, delta = ("ADD" if d > 0 else "TRIM" if d < 0 else "HOLD"), d
        rec = {"ticker": ticker, "company": cells[1].strip(),
               "latest_stake": latest, "prev_stake": prev, "move": move,
               "delta": delta, "value_cr": _value_cr(cells[2]), "qty": cells[3].strip()}
        rec.update({q: v for q, v in zip(quarters, series)})   # keep the full per-quarter stake trail
        out.append(rec)
    return out, True


MASTER_MIN_STOCKS = 50        # fewer unique stocks than this ⇒ scrape looks broken ⇒ don't overwrite
MASTER_MIN_SUCCESS = 0.60     # need ≥60% of investors fetched OK, else broken (e.g. a site-wide block)


def aggregate_master(rows):
    """Build the master_stock DataFrame from scraped holding `rows`, or return (None, msg)
    if the scrape looks broken (so the caller skips the write)."""
    ok_inv = {r["_investor"] for r in rows if r.get("_status") == "ok"}
    empty_inv = {r["_investor"] for r in rows if r.get("_status") == "empty"}
    fail_inv = {r["_investor"] for r in rows if r.get("_status") == "fail"}
    attempted = len(ok_inv | empty_inv | fail_inv)
    succ_rate = (attempted - len(fail_inv)) / max(1, attempted)
    hold_rows = [r for r in rows if r.get("_status") == "ok" and str(r.get("ticker", "")).strip()]
    n_uniq = len({r["ticker"] for r in hold_rows})
    if not hold_rows or n_uniq < MASTER_MIN_STOCKS or succ_rate < MASTER_MIN_SUCCESS:
        return None, (f"BROKEN scrape (uniq stocks={n_uniq}, success={succ_rate:.0%}, "
                      f"fails={len(fail_inv)}/{attempted})")
    H = pd.DataFrame(hold_rows)
    H = H[H["latest_stake"].notna()]                          # CURRENTLY held (latest-quarter stake present)

    def _holders(s):
        u = sorted(set(s))
        return ", ".join(u[:8]) + (" …" if len(u) > 8 else "")

    def _action(s):
        c = pd.Series(list(s)).value_counts()
        parts = [f"{int(c[k])} {lbl}" for k, lbl in (("NEW", "new"), ("ADD", "added"), ("TRIM", "trimmed"))
                 if c.get(k)]
        return " · ".join(parts)

    master = (H.groupby("ticker").agg(
        company=("company", "first"),
        superstar_count=("_investor", "nunique"),
        held_by=("_investor", _holders),
        total_value_cr=("value_cr", lambda s: round(float(pd.to_numeric(s, errors="coerce").fillna(0).sum()), 1)),
        recent_action=("move", _action),
    ).reset_index())
    master = master[master["ticker"].astype(str).str.strip() != ""]
    master = master.sort_values(["superstar_count", "total_value_cr"],
                                ascending=False).reset_index(drop=True)
    master.insert(0, "as_of", datetime.now().strftime("%Y-%m-%d"))
    return master, f"OK ({len(master)} stocks, success {succ_rate:.0%})"


def build_master_stock(sheet_key, links_tab="indian_fii_dii_trend_lyne_links",
                       master_tab="master_stock", checkpoint_file=None):
    """Daily superstar universe = union of EVERY investor's current holdings. Checkpointed
    (resume-safe on Ctrl-C / re-run); FAILED investors are RETRIED on the next run (only
    successes/empties are skipped). Overwrites `master_tab` ONLY when the scrape is healthy;
    on a broken scrape it leaves the existing tab UNTOUCHED and prints an ALERT."""
    inv = read_sheet(sheet_key, links_tab)
    if inv.empty or "links" not in inv.columns or "name" not in inv.columns:
        print(f"ALERT: no investor links in '{links_tab}' — master_stock NOT written."); return
    inv = inv[inv["links"].astype(str).str.startswith("http")].reset_index(drop=True)
    total = len(inv)
    if checkpoint_file is None:
        checkpoint_file = os.path.join(os.getcwd(), f"{master_tab}_checkpoint.csv")

    rows, done = [], set()
    if os.path.exists(checkpoint_file):
        try:
            ck = pd.read_csv(checkpoint_file)
            rows = ck.where(ck.notna(), None).to_dict("records")
            rows = [r for r in rows if str(r.get("_status")) != "fail"]          # drop fails → retry them
            done = {str(r.get("_investor", "")) for r in rows if str(r.get("_status")) in ("ok", "empty")}
            print(f"Resuming: {len(done)} investors done; previous failures will be RETRIED.")
        except Exception as e:
            print(f"Checkpoint load failed ({e}); starting fresh."); rows, done = [], set()

    sess = _make_session("https://trendlyne.com")   # warmed browser session (cookies + full headers) = anti-block
    try:                                            # 2nd warm-up hit + settle so the FIRST investor isn't sacrificial
        time.sleep(MASTER_DELAY)
        sess.get("https://trendlyne.com/portfolio/superstar-shareholders/index/", timeout=20)
        time.sleep(MASTER_DELAY)
    except Exception:
        pass
    delay = MASTER_DELAY
    print(f"Scraping holdings for {total} investors "
          f"(~{(MASTER_DELAY + MASTER_JITTER / 2) * (total - len(done)) / 60:.0f} min remaining "
          f"at {MASTER_DELAY:.0f}s+jitter each)...\n")
    consec_fail = 0
    try:
        for i, r in inv.iterrows():
            name = str(r.get("name", "") or "").strip()
            url = str(r.get("links", "") or "").strip()
            if not name or name in done:
                continue
            hold, ok = fetch_holdings(url, sess, delay=delay)
            if not ok:
                rows.append({"_investor": name, "_status": "fail", "ticker": ""})
                consec_fail += 1
                print(f"[{i + 1:>3}/{total}] {name[:38]:<38} FAIL (fetch/parse)")
                if consec_fail >= 4:                       # burst of fails ⇒ likely throttled → cool down + fresh session
                    delay = min(delay * 1.5, MASTER_MAX_DELAY)            # back off the request RATE, not just pause
                    print(f"    ⏸  {consec_fail} consecutive fails — cooling 90s, re-warming session, "
                          f"and slowing to {delay:.0f}s/req (likely rate-limited)…")
                    time.sleep(90); sess = _make_session("https://trendlyne.com"); consec_fail = 0
            elif not hold:
                rows.append({"_investor": name, "_status": "empty", "ticker": ""}); consec_fail = 0
                print(f"[{i + 1:>3}/{total}] {name[:38]:<38} 0 holdings")
            else:
                for h in hold:
                    rows.append({"_investor": name, "_status": "ok", **h})
                consec_fail = 0
                print(f"[{i + 1:>3}/{total}] {name[:38]:<38} {len(hold)} holdings")
            done.add(name)
            try:
                pd.DataFrame(rows).to_csv(checkpoint_file, index=False)
            except Exception:
                pass
    except KeyboardInterrupt:
        print(f"\nInterrupted — progress saved. Re-run to resume (failures will be retried).")
        return

    master, msg = aggregate_master(rows)
    if master is None:
        print(f"\n🚨 ALERT: {msg} — master_stock LEFT UNCHANGED (existing data preserved). "
              "Investigate (site block / layout change) and re-run.")
        return

    write_sheet(sheet_key, master_tab, master)                # also skips an empty df (belt-and-braces)
    print(f"\n✅ master_stock written — {msg}.")
    print(master.head(15).to_string(index=False))

    # also write the raw per-investor MOVES this quarter (NEW/ADD/TRIM/EXIT) → powers the dashboard
    # Alerts page ("which companies did the BEST investors newly buy?"). No extra scraping — reuse `rows`.
    _mv = [r for r in rows if r.get("_status") == "ok" and r.get("move") in ("NEW", "ADD", "TRIM", "EXIT")
           and str(r.get("ticker", "")).strip()]
    if _mv:
        mv = pd.DataFrame(_mv)[["_investor", "company", "ticker", "move", "latest_stake",
                                "prev_stake", "delta", "value_cr"]]
        mv.columns = ["investor", "company", "ticker", "move", "latest_stake", "prev_stake", "delta", "value_cr"]
        mv.insert(0, "as_of", datetime.now().strftime("%Y-%m-%d"))
        mv["_o"] = mv["move"].map({"NEW": 0, "ADD": 1, "TRIM": 2, "EXIT": 3}).fillna(9)
        mv = mv.sort_values(["_o", "investor"]).drop(columns="_o").reset_index(drop=True)
        write_sheet(sheet_key, "superstar_moves", mv)
        print(f"superstar_moves written: {len(mv)} moves ({int((mv['move'] == 'NEW').sum())} new).")
    # also write the per-investor HOLDINGS JOURNEY (full multi-quarter stake trail) so the app's
    # superstar detail page reads it here instead of live-scraping Trendlyne (blocked on Cloud).
    _hj = [r for r in rows if r.get("_status") == "ok" and str(r.get("ticker", "")).strip()]
    if _hj:
        _MON = ("Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec")
        def _is_q(c):
            p = str(c).split()
            return len(p) == 2 and p[0] in _MON and p[1].isdigit() and len(p[1]) == 4
        def _qk(c):
            try: return pd.Timestamp(str(c))
            except Exception: return pd.Timestamp("1900-01-01")
        hjd = pd.DataFrame(_hj).rename(columns={"_investor": "investor"})
        _qc = sorted([c for c in hjd.columns if _is_q(c)], key=_qk, reverse=True)
        _meta = [c for c in ("investor", "ticker", "company", "move", "delta", "value_cr", "qty") if c in hjd.columns]
        hjd = hjd[_meta + _qc]
        hjd.insert(0, "as_of", datetime.now().strftime("%Y-%m-%d"))
        write_sheet(sheet_key, "superstar_holdings", hjd)
        print(f"superstar_holdings written: {len(hjd)} positions across {hjd['investor'].nunique()} investors.")
    try:
        if os.path.exists(checkpoint_file):
            os.remove(checkpoint_file)
            print("Checkpoint deleted (run complete).")
    except Exception:
        pass

print("Master-stock builder ready.")

Master-stock builder ready.


In [15]:
# ── CELL 14: BUILD MASTER STOCK (run daily, after Cell 6 + Cell 13) ───────
build_master_stock(SHEET_KEY)

Scraping holdings for 254 investors (~21 min remaining at 4s+jitter each)...

[  1/254] ajay upadhyaya                         25 holdings
[  2/254] akash bhanshali                        22 holdings
[  3/254] amit gupta                             13 holdings
[  4/254] anil kumar goel and associates         39 holdings
[  5/254] anuj anantrai sheth and associates     7 holdings
[  6/254] ashish dhawan                          16 holdings
[  7/254] ashish kacholia                        70 holdings
[  8/254] ashok kumar jain                       50 holdings
[  9/254] atim kabra                             3 holdings
[ 10/254] bharat jayantilal patel and associates 15 holdings
[ 11/254] dheeraj kumar lohia and associates     70 holdings
[ 12/254] dilipkumar lakhi                       11 holdings
[ 13/254] dipak kanayalal shah                   65 holdings
[ 14/254] dolly khanna                           34 holdings
[ 15/254] harsha hitesh javeri                   34 holdings
[ 16/254]

In [16]:
# ── CELL 15: SUPERSTAR SAST (Reg 29) MOVES — near-real-time buy/sell tracker ──
# SEBI SAST Reg 29 disclosures catch when an investor crosses 5% (Reg29(1)) or moves +/-2%
# (Reg29(2)) in a stock — the signal that catches slow accumulation that BULK/BLOCK deals miss.
# One structured NSE JSON endpoint: acquirer name, stock, Acquisition/Sale, % traded, resulting
# stake %, Reg type, dates. We match acquirer names to the superstar list and write the matches.
import requests as _rq
from datetime import timedelta as _td

def _nse_session():
    h={'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
       'Accept':'text/html,*/*;q=0.8','Accept-Language':'en-US,en;q=0.9','Accept-Encoding':'gzip, deflate'}
    s=_rq.Session(); s.headers.update(h)
    try:
        s.get('https://www.nseindia.com', timeout=25); time.sleep(1.5)
        s.get('https://www.nseindia.com/companies-listing/corporate-filings-regulation-29', timeout=25); time.sleep(1.5)
    except Exception:
        pass
    return s

def fetch_sast_reg29(days=7, retries=4):
    """Fetch recent SEBI SAST Reg 29 disclosures from NSE as a DataFrame (empty df on block/failure)."""
    to=datetime.now(); frm=to - _td(days=days)
    aj={'Accept':'application/json,*/*;q=0.01','X-Requested-With':'XMLHttpRequest',
        'Referer':'https://www.nseindia.com/companies-listing/corporate-filings-regulation-29'}
    s=_nse_session()
    url=(f"https://www.nseindia.com/api/corporate-sast-reg29?index=equities"
         f"&from_date={frm.strftime('%d-%m-%Y')}&to_date={to.strftime('%d-%m-%Y')}")
    for a in range(retries):
        try:
            r=s.get(url, timeout=30, headers={**s.headers, **aj})
            if r.status_code==200:
                j=r.json(); d=j if isinstance(j, list) else j.get('data', [])
                if isinstance(d, list) and (not d or 'application_no' in d[0]):
                    return pd.DataFrame(d)
            time.sleep(6*(a+1))
        except Exception as e:
            print(f'    SAST fetch failed: {e}'); time.sleep(6*(a+1))
    return pd.DataFrame()

_SAST_SUF={'PVT','PRIVATE','LIMITED','LTD','LLP','HUF','AND','ASSOCIATES','SONS','CO','COMPANY',
           'THE','INDIA','MR','MRS','MS','GROUP','ESTATE','OF','LATE','SHRI'}
def _sast_toks(x):
    return [t for t in re.sub(r'[^A-Z0-9 ]', ' ', str(x).upper()).split() if t and t not in _SAST_SUF]

# Curated aliases: Trendlyne calls a house "icici group" but NSE files as "ICICI Prudential Mutual
# Fund" etc. — token matching can't bridge that, so we map each to its INVESTING arm(s) here.
# Editable via the superstar_sast_aliases sheet tab (investor · alias) — your review loop.
_SAST_ALIAS_SEED = {
    'icici group':          ['ICICI PRUDENTIAL', 'ICICI BANK'],
    'hdfc group':           ['HDFC MUTUAL FUND', 'HDFC LIFE', 'HDFC AMC'],
    'sbi group':            ['SBI MUTUAL FUND', 'SBI LIFE', 'SBI FUNDS'],
    'axis group':           ['AXIS MUTUAL FUND'],
    'kotak mahindra group': ['KOTAK MAHINDRA MUTUAL FUND', 'KOTAK MAHINDRA BANK'],
    'birla group':          ['ADITYA BIRLA SUN LIFE', 'ABSL', 'ADITYA BIRLA CAPITAL'],
}

def load_sast_aliases(sheet_key, tab='superstar_sast_aliases'):
    """Read investor->alias patterns from the sheet (seed the tab from _SAST_ALIAS_SEED if empty)."""
    df=read_sheet(sheet_key, tab)
    if df.empty or not {'investor','alias'}.issubset(df.columns):
        seed=pd.DataFrame([{'investor':k,'alias':a} for k,pats in _SAST_ALIAS_SEED.items() for a in pats])
        write_sheet(sheet_key, tab, seed); df=seed
    out={}
    for _,r in df.iterrows():
        inv=str(r['investor']).strip().lower(); al=str(r['alias']).strip().upper()
        if inv and al and al!='NAN': out.setdefault(inv, []).append(al)
    return out

def match_superstar_sast(sast, names_df, aliases=None):
    """Match each SAST acquirerName to a superstar. Curated `aliases` (investor->[UPPER substrings])
    win first as HIGH; otherwise fall back to token-subset name matching (HIGH / REVIEW)."""
    aliases=aliases or {}
    name_by_lower={str(n).strip().lower(): str(n) for n in names_df['name'].astype(str)}
    names=[(n, set(_sast_toks(n))) for n in names_df['name'].astype(str) if len(_sast_toks(n))>=2]
    typ=dict(zip(names_df['name'], names_df['type'])) if 'type' in names_df.columns else {}
    out=[]
    for _, r in sast.iterrows():
        an=str(r.get('acquirerName', '')); au=an.upper(); at=set(_sast_toks(an))
        nm=conf=None
        for inv_l, pats in aliases.items():             # curated alias map wins (trusted)
            if inv_l in name_by_lower and any(p in au for p in pats):
                nm, conf = name_by_lower[inv_l], 'HIGH'; break
        if nm is None and at:
            for cand, it in names:
                if it<=at:                               # all investor tokens present in the acquirer name
                    conf='HIGH' if (it==at or len(it)>=3 or len(at)<=len(it)+1) else 'REVIEW'
                    nm=cand; break
        if nm:
            buy=str(r.get('acqSaleType', ''))=='Acquisition'
            acq=pd.to_numeric(r.get('totAcqShare'), errors='coerce')
            sal=pd.to_numeric(r.get('totSaleShare'), errors='coerce')
            out.append({'filed': r.get('timestamp'), 'trade_dates': r.get('acquirerDate'), 'investor': nm,
                        'inv_type': typ.get(nm, ''), 'acquirer_name': r.get('acquirerName'),
                        'symbol': r.get('symbol'), 'company': r.get('company'), 'action': r.get('acqSaleType'),
                        'pct_traded': acq if buy else sal, 'pct_after': pd.to_numeric(r.get('totAftShare'), errors='coerce'),
                        'reg_type': r.get('regType'), 'promoter': r.get('promoterType'), 'confidence': conf,
                        'application_no': r.get('application_no'), 'attachment': r.get('attachement')})
    return pd.DataFrame(out)

def build_superstar_sast(sheet_key, days=7, deals_tab='superstar_sast_deals'):
    """Fetch recent SAST Reg29 disclosures, match to superstars, append+dedup into the deals tab.
    Block-safe: if the NSE fetch returns nothing, the existing tab is left untouched."""
    names=read_sheet(sheet_key, 'fii_dii_indian_investment_summary')
    if names.empty or 'name' not in names.columns:
        print('  SAST: no investor summary to match against — skipped.'); return
    sast=fetch_sast_reg29(days=days)
    if sast.empty:
        print('  ALERT: SAST fetch returned nothing (NSE likely blocked/rate-limited this IP) — tab preserved.'); return
    aliases=load_sast_aliases(sheet_key)
    m=match_superstar_sast(sast, names, aliases)
    print(f'  SAST: {len(sast)} disclosures -> {len(m)} superstar matches '
          f'({m["investor"].nunique() if len(m) else 0} investors).')
    if m.empty:
        print('  (no superstar matches in this window; nothing written.)'); return
    prev=read_sheet(sheet_key, deals_tab)
    combined=pd.concat([prev, m], ignore_index=True) if not prev.empty else m
    # dedup on the EVENT signature (NSE re-files the same disclosure under different application_no)
    _dk=[c for c in ['acquirer_name','symbol','trade_dates','action'] if c in combined.columns]
    if _dk: combined=combined.drop_duplicates(subset=_dk, keep='last')
    combined['_ord']=pd.to_datetime(combined['filed'], errors='coerce')
    combined=combined.sort_values('_ord', ascending=False).drop(columns=['_ord']).reset_index(drop=True)
    write_sheet(sheet_key, deals_tab, combined)
    n_rev=int((combined['confidence']=='REVIEW').sum()) if 'confidence' in combined.columns else 0
    print(f'  superstar_sast_deals written: {len(combined)} rows ({n_rev} tagged REVIEW).')

print('SAST Reg29 builder ready.')


SAST Reg29 builder ready.


In [17]:
# ── CELL 16: BUILD SUPERSTAR SAST (run daily, after the summary is fresh) ──
build_superstar_sast(SHEET_KEY, days=7)


  SAST: 52 disclosures -> 2 superstar matches (2 investors).
  Wrote 78 rows to "superstar_sast_deals".
  superstar_sast_deals written: 78 rows (10 tagged REVIEW).


In [18]:
# ── CELL 17: SUPERSTAR BULK & BLOCK DEALS — via Trendlyne per-investor pages ──
# Trendlyne clusters every investor's deals (BSE+NSE, all entities/AIFs/family accounts) under one
# investor ID (from `links`). We scrape each investor's bulk-block page by ID — no name-matching,
# no BSE gap, all HIGH-confidence. Capture the ticker + clean company from the stock link, and
# normalise the date to ISO so downstream analysis (round-trips, % move) is reliable.
def _tl_id_slug(link):
    m=re.search(r'/superstar-shareholders/(\d+)/[^/]+/([^/?]+)', str(link))
    return (m.group(1), m.group(2)) if m else (None, None)

def _tl_norm_date(x):
    try:
        return pd.to_datetime(str(x).strip(), dayfirst=True).strftime('%Y-%m-%d')
    except Exception:
        return str(x).strip()

def _pct_num(x):
    """'1.83%' / '(18.58%)' -> 1.83 / 18.58 (float percent), or None. Stored as a real number so
    Google Sheets doesn't convert the '%' string to a 0-1 fraction."""
    m=re.search(r'-?\d+\.?\d*', str(x))
    return float(m.group()) if m else None

def _tl_stock_meta(href):
    """From a Trendlyne stock link -> (ticker, clean_company). Ticker may be a numeric BSE code."""
    m=re.search(r'/equity/[^/]+/([^/]+)/\d+/([^/?]+)', str(href))
    if not m:
        return '', ''
    tk=m.group(1).strip().upper()
    comp=m.group(2).replace('-', ' ').strip().title()
    return tk, comp

def _tl_scrape_bbdeals(session, url, retries=3):
    """Scrape one investor's bulk/block table -> list of dicts. None on fetch failure (block)."""
    for a in range(retries):
        try:
            time.sleep(REQUEST_DELAY + np.random.uniform(0, 1.0))
            r=session.get(url, timeout=25)
            if r.status_code==200:
                tb=BeautifulSoup(r.text, 'html.parser').find('table')
                if tb is None:
                    return []
                heads=[th.get_text(' ', strip=True) for th in tb.find_all('th')]
                out=[]
                for tr in (tb.find('tbody') or tb).find_all('tr'):
                    tds=tr.find_all('td')
                    if not tds:
                        continue
                    cells=[td.get_text(' ', strip=True) for td in tds]
                    if not any(cells):
                        continue
                    rec=dict(zip(heads, cells))
                    aa=tds[0].find('a', href=True)
                    rec['__href']=aa['href'] if aa else ''
                    out.append(rec)
                return out
            time.sleep(5*(a+1))
        except Exception:
            time.sleep(5*(a+1))
    return None

def build_superstar_bulkblock(sheet_key, deals_tab='superstar_bulkblock_deals'):
    """Scrape each superstar's Trendlyne bulk/block page (by ID) -> deals tab. All HIGH-confidence.
    Block-safe: skips the write if <40% of investors fetched."""
    inv=read_sheet(sheet_key, 'fii_dii_indian_investment_summary')
    if inv.empty or 'name' not in inv.columns or 'links' not in inv.columns:
        print('  bulk/block: summary missing name/links — skipped.'); return
    typ=dict(zip(inv['name'], inv['type'])) if 'type' in inv.columns else {}
    sess=_make_session('https://trendlyne.com')
    rows=[]; ok=fail=0; total=len(inv)
    print(f'Scraping bulk/block for {total} investors via Trendlyne...')
    for i, r in inv.iterrows():
        iid, slug=_tl_id_slug(r.get('links', ''))
        if not iid:
            continue
        recs=_tl_scrape_bbdeals(sess, f'https://trendlyne.com/portfolio/bulk-block-deals/{iid}/{slug}/')
        if recs is None:
            fail+=1; continue
        ok+=1
        for d in recs:
            act=str(d.get('Action', '')).strip().upper()
            tk, comp=_tl_stock_meta(d.get('__href', ''))
            rows.append({'date': _tl_norm_date(d.get('Date')), 'investor': r['name'], 'inv_type': typ.get(r['name'], ''),
                         'ticker': tk, 'company': comp or d.get('Stock'), 'entity': d.get('Client Name'),
                         'exchange': d.get('Exchange'), 'deal_type': d.get('Deal Type'),
                         'action': 'BUY' if act.startswith('PUR') else ('SELL' if act.startswith('SEL') else act),
                         'qty': str(d.get('Quantity', '')).replace(',', ''),
                         'price': str(d.get('Avg. Price', '')).replace(',', ''),
                         'pct_traded': _pct_num(d.get('Percentage Traded %')), 'confidence': 'HIGH'})
    succ=ok/max(1, ok+fail)
    print(f'  fetched {ok}/{ok+fail} investors ({succ:.0%}); {len(rows)} deals.')
    if succ < 0.4 or not rows:
        print(f'  ALERT: bulk/block scrape looks blocked ({succ:.0%}) — "{deals_tab}" LEFT UNCHANGED.'); return
    out=pd.DataFrame(rows)
    out=out.sort_values('date', ascending=False).reset_index(drop=True)   # ISO dates sort correctly
    write_sheet(sheet_key, deals_tab, out)
    print(f'  superstar_bulkblock_deals written: {len(out)} deals · {out["investor"].nunique()} investors.')

print('Bulk/block (Trendlyne) builder ready.')


Bulk/block (Trendlyne) builder ready.


In [19]:
# ── CELL 18: BUILD SUPERSTAR BULK/BLOCK (run daily) ──
build_superstar_bulkblock(SHEET_KEY)


Scraping bulk/block for 186 investors via Trendlyne...
  fetched 185/186 investors (99%); 10908 deals.
  Wrote 10908 rows to "superstar_bulkblock_deals".
  superstar_bulkblock_deals written: 10908 deals · 178 investors.


In [20]:
# ── CELL 19: SUPERSTAR INSIDER / SAST DEALS — via Trendlyne per-investor pages ──
# Same per-investor-ID approach as bulk/block: scrape each investor's insider-trading-sast page.
# Covers promoter/insider (PIT) + SAST threshold disclosures, clustered across all their entities.
def _ins_stock_meta(href):
    m=re.search(r'/insider-trading-sast/(?:all/)?([^/]+)/\d+/([^/?]+)', str(href))
    if not m:
        return '', ''
    return m.group(1).strip().upper(), m.group(2).replace('-', ' ').strip().title()

def _ins_pct(x):
    m=re.search(r'\(([-\d.]+)%\)', str(x))
    return float(m.group(1)) if m else None

def build_superstar_insider(sheet_key, deals_tab='superstar_insider_deals'):
    """Scrape each superstar's Trendlyne insider/SAST page (by ID) -> deals tab. All HIGH-confidence.
    Block-safe: skips the write if <40% of investors fetched."""
    inv=read_sheet(sheet_key, 'fii_dii_indian_investment_summary')
    if inv.empty or 'name' not in inv.columns or 'links' not in inv.columns:
        print('  insider: summary missing name/links — skipped.'); return
    typ=dict(zip(inv['name'], inv['type'])) if 'type' in inv.columns else {}
    sess=_make_session('https://trendlyne.com')
    rows=[]; ok=fail=0; total=len(inv)
    print(f'Scraping insider/SAST for {total} investors via Trendlyne...')
    for i, r in inv.iterrows():
        iid, slug=_tl_id_slug(r.get('links', ''))
        if not iid:
            continue
        recs=_tl_scrape_bbdeals(sess, f'https://trendlyne.com/equity/insider-trading-sast/{iid}/{slug}/')
        if recs is None:
            fail+=1; continue
        ok+=1
        for d in recs:
            tk, comp=_ins_stock_meta(d.get('__href', ''))
            rows.append({'report_date': _tl_norm_date(d.get('Reported To/By Exchange')),
                         'investor': r['name'], 'inv_type': typ.get(r['name'], ''),
                         'ticker': tk, 'company': comp or d.get('Stock'),
                         'person': d.get('Client Name'), 'category': d.get('Client Category'),
                         'action': d.get('Action*'), 'qty': str(d.get('Quantity', '')).replace(',', ''),
                         'holding_after': _ins_pct(d.get('Post Transaction Holding')),
                         'traded_pct': _pct_num(d.get('Traded %')), 'regulation': d.get('Regulation (Insider/SAST)'),
                         'trade_period': d.get('Period'), 'confidence': 'HIGH'})
    succ=ok/max(1, ok+fail)
    print(f'  fetched {ok}/{ok+fail} investors ({succ:.0%}); {len(rows)} insider/SAST rows.')
    if succ < 0.4 or not rows:
        print(f'  ALERT: insider scrape looks blocked ({succ:.0%}) — "{deals_tab}" LEFT UNCHANGED.'); return
    out=pd.DataFrame(rows).sort_values('report_date', ascending=False).reset_index(drop=True)
    write_sheet(sheet_key, deals_tab, out)
    print(f'  superstar_insider_deals written: {len(out)} rows · {out["investor"].nunique()} investors.')

print('Insider/SAST (Trendlyne) builder ready.')


Insider/SAST (Trendlyne) builder ready.


In [21]:
# ── CELL 20: BUILD SUPERSTAR INSIDER/SAST (run daily) ──
build_superstar_insider(SHEET_KEY)


Scraping insider/SAST for 186 investors via Trendlyne...
  fetched 185/186 investors (99%); 2070 insider/SAST rows.
  Wrote 2070 rows to "superstar_insider_deals".
  superstar_insider_deals written: 2070 rows · 115 investors.


In [22]:
# ── CELL 21: MARKET-WIDE BULK & BLOCK DEALS (all-market, recent) ──
# For the dashboard "📅 Today's deals" browser — the latest NSE bulk+block across the WHOLE market
# (not just superstars). Uses the CSV export (&csv=true) = full data, range honored. Block-safe.
def build_market_bulkblock(sheet_key, days=7, tab='market_bulkblock_deals'):
    import io as _io2
    to=datetime.now(); frm=to - _td(days=days)
    ref='https://www.nseindia.com/report-detail/display-bulk-and-block-deals'
    aj={'Accept':'*/*', 'X-Requested-With':'XMLHttpRequest', 'Referer':ref}
    s=_nse_session()
    try:
        s.get(ref, timeout=25); time.sleep(1.0)
    except Exception:
        pass
    frames=[]
    for opt, label in (('bulk_deals','bulk'), ('block_deals','block')):
        url=(f'https://www.nseindia.com/api/historicalOR/bulk-block-short-deals'
             f'?optionType={opt}&from={frm.strftime("%d-%m-%Y")}&to={to.strftime("%d-%m-%Y")}&csv=true')
        for a in range(4):
            try:
                r=s.get(url, timeout=45, headers={**s.headers, **aj})
                if r.status_code==200 and 'SYMBOL' in r.text[:150].upper():
                    df=pd.read_csv(_io2.BytesIO(r.content), encoding='utf-8-sig')
                    df.columns=[c.strip() for c in df.columns]; df['deal_type']=label; frames.append(df); break
                time.sleep(6*(a+1))
            except Exception as e:
                print(f'  market {opt} fetch failed: {e}'); time.sleep(6*(a+1))
    if not frames:
        print('  ALERT: market bulk/block fetch returned nothing (NSE blocked?) — tab preserved.'); return
    d=pd.concat(frames, ignore_index=True).rename(columns={
        'Date':'date', 'Symbol':'symbol', 'Security Name':'company', 'Client Name':'client',
        'Buy / Sell':'action', 'Quantity Traded':'qty', 'Trade Price / Wght. Avg. Price':'price'})
    d=d[[c for c in ['date','symbol','company','client','action','qty','price','deal_type'] if c in d.columns]]
    for c in d.columns:
        if d[c].dtype==object:
            d[c]=d[c].astype(str).str.strip()
    d['date']=d['date'].map(_tl_norm_date)
    d=d.sort_values('date', ascending=False).reset_index(drop=True)
    write_sheet(sheet_key, tab, d)
    print(f'  market_bulkblock_deals written: {len(d)} deals across {d["date"].nunique()} day(s).')

print('Market-wide bulk/block builder ready.')


Market-wide bulk/block builder ready.


In [23]:
# ── CELL 22: BUILD MARKET-WIDE BULK/BLOCK (run daily) ──
build_market_bulkblock(SHEET_KEY, days=7)


  Wrote 769 rows to "market_bulkblock_deals".
  market_bulkblock_deals written: 769 deals across 6 day(s).
